# Visual-Doc Assistant — Evaluation 1: System Evaluation

This notebook implements **Evaluation 1** from the eval plan:

> "How well does my Visual-Doc Assistant retrieve and answer?"

It has two stages, run against the exact same index your Streamlit app (`app.py`, Step 9D) already built:

1. **Retrieval** — Does ColPali + **full late interaction (exact MaxSim)** retrieve the correct page? Metrics: Recall@1, Recall@3, Recall@5, Recall@10.
2. **Generation** — Given the retrieved page(s), does Gemini produce the correct answer? Metric: answer accuracy (keyword-grounded, see the Generation section for how to swap in an LLM-judge instead).

**Retrieval mode is locked to `exhaustive`** — every one of the ~521 indexed pages across all 3 documents is scored with the true MaxSim formula, zero ANN-shortlist approximation anywhere. This notebook does **not** touch, import, or fall back to any mean-pooled cosine scoring path from Steps 1-9C.

**Corpus:** all 3 currently indexed documents, evaluated as a **single combined ~521-page index** — questions are *not* pre-filtered to their source document. This is the harder, more realistic test: two of the three documents (`WorldDevelopmentReport`, `AIFoundations`) are both World Bank digital-economy reports, so there's real potential for cross-document confusion that a per-document-filtered eval would hide.

Run cells top to bottom. Cells 2-4 assume this notebook lives in the same Colab session / Drive path as your Phase 3 `app.py` (same `DB_PATH`, same `IMAGE_FOLDER`) so it reuses the *already-computed* patch embeddings on disk — it does not re-embed anything unless a page turns out not to be MaxSim-ready (see Cell 5).

---
**Completion additions (Sections 5b, 6b-6d, 7b, 8b).** The original cells and results above are unchanged. These sections add what the evaluation plan promised but the first version did not include, so that Evaluation 2 (OCR/text RAG vs. ColPali) starts from a finished, frozen baseline: MRR and nDCG@5, efficiency (latency, storage, indexing time), the retrieval-mode ablation, a scoring audit with faithfulness and source-page accuracy, optional extra text-heavy questions, and a frozen benchmark/config/metric module.

---
**Final-pass edits (CPU-friendly).** Section 6 can reuse the saved retrieval results instead of re-running the 8-minute CPU loop; 6c and 6d now run on a plain CPU runtime (no T4 needed) and label which numbers are device-dependent; the abstention patterns were widened; hand-grade overrides are set; 8b records library versions, judge models and Wilson 95% intervals and no longer silently skips missing efficiency/ablation results.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Setup — same config as Step 9D, no Streamlit dependency

In [2]:
!pip install -q -U colpali-engine chromadb pdf2image google-genai "torchao>=0.16.0" "peft>=0.14.0"
!apt-get -qq install -y poppler-utils > /dev/null

import os
import numpy as np
import torch
import chromadb
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor

# ---- Same paths/constants as app.py (Step 9D) -- point these at your Drive mount ----
MODEL_NAME   = "vidore/colpali-v1.2"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_FOLDER = "/content/drive/MyDrive/Visual-Doc/processed_images"
DB_PATH      = "/content/drive/MyDrive/Visual-Doc/chroma_db_storage"

# If running in Colab and Drive isn't mounted yet:
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except ImportError:
    pass

print(f"Device: {DEVICE}")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


W1119 10:24:20.056000 529 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1119 10:24:20.124000 529 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Device: cpu


In [3]:
# ---- Load model + collection (plain functions, no @st.cache_resource) ----
print("Loading ColPali model (first load only takes a while)...")
model = ColPali.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map=DEVICE).eval()
processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

db_client = chromadb.PersistentClient(path=DB_PATH)
collection = db_client.get_or_create_collection(
    name="visual_doc_collection", metadata={"hnsw:space": "cosine"}
)
print(f"Collection loaded -- {collection.count()} pages indexed total.")


Loading ColPali model (first load only takes a while)...


Collection loaded -- 521 pages indexed total.


## 2. Core scoring functions -- copied from Step 9D's `app.py` (MaxSim path only)

These are the *exact same* `embed_query`, `maxsim_score`, and readiness-check functions as your Streamlit app -- just with the `@st.cache_*` decorators removed (plain Python instead) since this runs outside Streamlit. **No mean-pooled-only retrieval path from Steps 1-9C is reproduced here at all** -- there is nothing in this notebook capable of silently scoring with the old approximation.

*(Exception: Section 6d, the retrieval-mode ablation, deliberately re-implements the mean-pooled and two-stage paths in isolated functions solely to compare them. Nothing in Sections 6-7 uses them.)*

In [4]:
@torch.no_grad()
def embed_query(query_text: str):
    """Returns (proxy, full_tokens) -- full_tokens is the TRUE per-token query
    embedding used for exact MaxSim scoring. Identical to Step 9D."""
    inputs = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)
    full_tokens = token_embeddings.squeeze(0).cpu().float().numpy()
    proxy = full_tokens.mean(axis=0)
    return proxy, full_tokens


def maxsim_score(query_tokens: np.ndarray, doc_patches: np.ndarray) -> float:
    """Exact ColPali/ColBERT late interaction: LI(q,d) = sum_i max_j <Eq(i)|Ed(j)>"""
    sims = query_tokens.astype(np.float32) @ doc_patches.astype(np.float32).T
    return float(sims.max(axis=1).sum())


def get_indexed_documents() -> dict:
    if collection.count() == 0:
        return {}
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    docs = {}
    for m in all_meta:
        key = (m.get("doc_name", "unknown"), m.get("doc_id_prefix", "unknown"))
        docs[key] = docs.get(key, 0) + 1
    return dict(sorted(docs.items(), key=lambda kv: kv[0][0].lower()))


def get_maxsim_readiness() -> dict:
    """{(doc_name, doc_id_prefix): (pages_with_full_embeddings, total_pages)}"""
    if collection.count() == 0:
        return {}
    all_meta = collection.get(include=["metadatas"])["metadatas"]
    stats = {}
    for m in all_meta:
        key = (m.get("doc_name", "unknown"), m.get("doc_id_prefix", "unknown"))
        ready, total = stats.get(key, (0, 0))
        total += 1
        patch_path = m.get("patch_embedding_path")
        if patch_path and os.path.exists(patch_path):
            ready += 1
        stats[key] = (ready, total)
    return stats


@torch.no_grad()
def embed_page_images_batch(images: list):
    """Same as Step 9D -- used only if upgrade_document_to_maxsim needs to backfill."""
    inputs = processor.process_images(images).to(DEVICE)
    batch_patch_embeddings = model(**inputs)
    results = []
    for patch_emb in batch_patch_embeddings:
        full32 = patch_emb.cpu().float().numpy()
        full = full32.astype(np.float16)
        proxy = full32.mean(axis=0).astype(np.float32)
        results.append((full, proxy))
    return results


def upgrade_document_to_maxsim(doc_id_prefix: str, batch_size: int = 4) -> dict:
    """Backfill full patch embeddings for any page missing them, from the page
    PNGs already on disk. Identical logic to Step 9D's version, minus the
    Streamlit progress bar."""
    matches = collection.get(where={"doc_id_prefix": doc_id_prefix}, include=["metadatas"])
    to_upgrade = [
        (mid, meta) for mid, meta in zip(matches["ids"], matches["metadatas"])
        if not (meta.get("patch_embedding_path") and os.path.exists(meta.get("patch_embedding_path", "")))
    ]
    total = len(to_upgrade)
    upgraded = 0
    for batch_start in range(0, total, batch_size):
        batch = to_upgrade[batch_start: batch_start + batch_size]
        batch_ids = [mid for mid, _ in batch]
        batch_metas = [meta for _, meta in batch]
        batch_images = [Image.open(meta["image_path"]).convert("RGB") for meta in batch_metas]
        embedded = embed_page_images_batch(batch_images)
        new_metas, new_embs = [], []
        for meta, (full, proxy) in zip(batch_metas, embedded):
            folder = os.path.dirname(meta["image_path"])
            page_num = meta["page_number"]
            patch_path = os.path.join(folder, f"page_{page_num}_patches.npy")
            np.save(patch_path, full)
            m2 = dict(meta)
            m2["patch_embedding_path"] = patch_path
            m2["num_patches"] = int(full.shape[0])
            new_metas.append(m2)
            new_embs.append(proxy.tolist())
        collection.update(ids=batch_ids, embeddings=new_embs, metadatas=new_metas)
        upgraded += len(batch)
    return {"upgraded": upgraded, "total": total}


## 3. MaxSim-readiness check

This is the precondition flagged during planning: `2407_01449v6` (the ColPali paper) was likely indexed back in Phase 1/2, before Step 9D's full-patch storage existed. If any page lacks a saved patch matrix, `retrieve_top_k_pages` would silently fall back to mean-pooled cosine for *that page only* -- which would quietly break the "full late interaction only" guarantee this eval is supposed to have. This cell checks every document and backfills anything missing before proceeding.

In [5]:
readiness = get_maxsim_readiness()
indexed_docs = get_indexed_documents()

print(f"{"Document":<30}{"MaxSim-ready pages":<22}{"Total pages"}")
needs_upgrade = []
for (doc_name, doc_id_prefix), total in indexed_docs.items():
    ready, _ = readiness.get((doc_name, doc_id_prefix), (0, total))
    print(f"{doc_name:<30}{ready:<22}{total}")
    if ready < total:
        needs_upgrade.append((doc_name, doc_id_prefix, total - ready))

if needs_upgrade:
    print()
    print("Upgrading documents with missing patch embeddings...")
    for doc_name, doc_id_prefix, missing in needs_upgrade:
        print(f"  {doc_name}: backfilling {missing} page(s)...")
        result = upgrade_document_to_maxsim(doc_id_prefix)
        print(f"    -> upgraded {result['upgraded']}/{result['total']}")
    print()
    print("All documents now MaxSim-ready.")
else:
    print()
    print("All documents already MaxSim-ready -- safe to proceed with exhaustive scoring.")


Document                      MaxSim-ready pages    Total pages
2407.01449v6                  26                    26
AIFoundations                 136                   136
WorldDevelopmentReport        359                   359

All documents already MaxSim-ready -- safe to proceed with exhaustive scoring.


## 4. Load the full corpus into memory once

Exhaustive MaxSim over ~521 pages needs every page's patch matrix loaded -- but only *once*, not once per query. This is the "cheap because it's cached" property discussed during planning: the expensive VLM forward pass already happened at indexing time, so this step is pure disk I/O (~130 MB total).

In [6]:
print("Loading all patch embeddings into memory...")
all_data = collection.get(include=["metadatas"])
CORPUS = []
for meta in all_data["metadatas"]:
    patch_path = meta.get("patch_embedding_path")
    if patch_path and os.path.exists(patch_path):
        CORPUS.append({
            "doc_name": meta["doc_name"],
            "page_number": meta["page_number"],
            "image_path": meta["image_path"],
            "patches": np.load(patch_path).astype(np.float32),
        })

total_mb = sum(c["patches"].nbytes for c in CORPUS) / 1e6
print(f"Loaded {len(CORPUS)} pages into memory ({total_mb:.1f} MB).")

DOC_NAME_COUNTS = {}
for c in CORPUS:
    DOC_NAME_COUNTS[c["doc_name"]] = DOC_NAME_COUNTS.get(c["doc_name"], 0) + 1
print(DOC_NAME_COUNTS)


def exhaustive_maxsim_search(query_text: str, top_k: int = 10):
    """Pure in-memory exhaustive MaxSim over the preloaded CORPUS -- no ANN
    shortlist, no disk I/O per call, no doc_filter (searches the full
    combined index across all documents)."""
    _, query_tokens = embed_query(query_text)
    scored = []
    for page in CORPUS:
        score = maxsim_score(query_tokens, page["patches"])
        scored.append((score, page))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [
        {"doc_name": p["doc_name"], "page_number": p["page_number"],
         "image_path": p["image_path"], "score": s}
        for s, p in scored[:top_k]
    ]


Loading all patch embeddings into memory...
Loaded 521 pages into memory (275.0 MB).
{'2407.01449v6': 26, 'AIFoundations': 136, 'WorldDevelopmentReport': 359}


## 5. The 30-question evaluation set

10 questions per document, page-grounded and hand-verified against the actual PDFs (not generated blind). Each question is tagged with a `modality` (chart / table / map / infographic / diagram / text) so results can be broken down by how visually demanding the source page is -- the `text` questions per document serve as a control: a page ColPali *shouldn't* need visual patches to answer, useful context once you build the OCR baseline (Evaluation 2).

In [7]:
QA_SET = [
    # ---------------- 2407_01449v6 (the ColPali paper, 26 pages) ----------------
    {"doc_name": "2407_01449v6", "page": 1, "modality": "text",
     "question": "What is the name of the benchmark introduced in this paper for evaluating page-level document retrieval?",
     "gold_answer": "ViDoRe (Visual Document Retrieval Benchmark)"},
    {"doc_name": "2407_01449v6", "page": 2, "modality": "chart",
     "question": "According to Figure 1, what is the offline indexing latency per page for ColPali, compared to the standard retrieval pipeline?",
     "gold_answer": "ColPali: 0.39s/page; standard retrieval: 7.22s/page"},
    {"doc_name": "2407_01449v6", "page": 4, "modality": "table",
     "question": "According to Table 1, how many queries and how many documents does the DocVQA dataset contain in ViDoRe?",
     "gold_answer": "500 queries and 500 documents"},
    {"doc_name": "2407_01449v6", "page": 5, "modality": "chart",
     "question": "According to Figure 2, what is the total per-page offline indexing latency for SigLIP?",
     "gold_answer": "0.12 seconds"},
    {"doc_name": "2407_01449v6", "page": 7, "modality": "table",
     "question": "According to Table 2, what nDCG@5 score does ColPali achieve on the TabFQuAD (TabF) benchmark?",
     "gold_answer": "83.9"},
    {"doc_name": "2407_01449v6", "page": 9, "modality": "chart",
     "question": "In the token pooling chart in Figure 3 (left panel), which ViDoRe dataset shows the most performance degradation as the pool factor increases?",
     "gold_answer": "Shift"},
    {"doc_name": "2407_01449v6", "page": 18, "modality": "table",
     "question": "According to Table 4, what is ColPali's per-page embedding size in KB (float16), and how does it compare to BM25's sparse embedding size?",
     "gold_answer": "ColPali: 257.5 KB; BM25 sparse: about 1.56 KB -- roughly two orders of magnitude smaller"},
    {"doc_name": "2407_01449v6", "page": 19, "modality": "table",
     "question": "According to Table 5, how long does the layout detection step take on average, per page, in the Unstructured pipeline?",
     "gold_answer": "0.81 seconds"},
    {"doc_name": "2407_01449v6", "page": 21, "modality": "table",
     "question": "According to Table 6, what is ColPali's average Recall@1 score across all ViDoRe tasks?",
     "gold_answer": "72.7"},
    {"doc_name": "2407_01449v6", "page": 22, "modality": "chart",
     "question": "In the similarity map example in Figure 6, which underlined query token is shown, and which offshore oil field also receives a high similarity score even though it is not the literal query word?",
     "gold_answer": "Query token: 'Kazakhstan'; offshore oil field: 'Kashagan'"},

    # ---------------- WorldDevelopmentReport (359 pages) ----------------
    {"doc_name": "WorldDevelopmentReport", "page": 36, "modality": "text",
     "question": "According to Box O.2, what share of households needed to be engaged in online commerce for a village to be considered a 'Taobao village'?",
     "gold_answer": "At least 10 percent"},
    {"doc_name": "WorldDevelopmentReport", "page": 78, "modality": "chart",
     "question": "According to Figure 1.2 panel b, which type of internet use had the highest adoption rate among high-income OECD firms in 2010-14?",
     "gold_answer": "E-mail (about 95 percent)"},
    {"doc_name": "WorldDevelopmentReport", "page": 83, "modality": "chart",
     "question": "According to Figure 1.7, in developing countries during 2010-14, which contributed more to growth: ICT capital or non-ICT capital?",
     "gold_answer": "Non-ICT capital"},
    {"doc_name": "WorldDevelopmentReport", "page": 128, "modality": "map",
     "question": "According to Map 2.1, what was the national mobile phone adoption rate in Sub-Saharan Africa, circa 2014?",
     "gold_answer": "63 percent"},
    {"doc_name": "WorldDevelopmentReport", "page": 132, "modality": "table",
     "question": "According to Table 2.2, what is the potential-impact rating for 'Connecting people to work and markets' on nonpoor consumers?",
     "gold_answer": "H (high)"},
    {"doc_name": "WorldDevelopmentReport", "page": 155, "modality": "chart",
     "question": "According to Figure 2.24, which country has the lowest adjusted share of employment susceptible to automation?",
     "gold_answer": "Uzbekistan"},
    {"doc_name": "WorldDevelopmentReport", "page": 198, "modality": "chart",
     "question": "According to Figure 3.14 panel a, around what year did the percentage of elections that are free and fair peak?",
     "gold_answer": "Around 1990 (about 77 percent)"},
    {"doc_name": "WorldDevelopmentReport", "page": 227, "modality": "infographic",
     "question": "According to Figure B4.1.1, how many billion people have high-speed internet access, out of the roughly 7.4 billion global population?",
     "gold_answer": "1.1 billion"},
    {"doc_name": "WorldDevelopmentReport", "page": 243, "modality": "chart",
     "question": "According to Figure 4.1, which of the four cost categories shown (storage, bandwidth, computing, smartphones) declined the least per year?",
     "gold_answer": "Smartphone cost (about -5 percent per year)"},
    {"doc_name": "WorldDevelopmentReport", "page": 285, "modality": "diagram",
     "question": "According to Figure 5.7, under which skill category does 'manual dexterity and the use of methods, materials, tools, and instruments' fall?",
     "gold_answer": "Technical"},

    # ---------------- AIFoundations (136 pages) ----------------
    {"doc_name": "AIFoundations", "page": 15, "modality": "text",
     "question": "According to the Main Messages section, what share of notable AI models do high-income countries (HICs) account for, as of July 2025?",
     "gold_answer": "87 percent"},
    {"doc_name": "AIFoundations", "page": 37, "modality": "map",
     "question": "According to Map 1.1 panel a, what is the highest ChatGPT-traffic band shown, in millions of visits?",
     "gold_answer": "482.9 million visits"},
    {"doc_name": "AIFoundations", "page": 40, "modality": "chart",
     "question": "According to Figure 1.4, which OECD country has the highest AI adoption rate among all firms (10+ employees)?",
     "gold_answer": "Korea (KOR)"},
    {"doc_name": "AIFoundations", "page": 53, "modality": "chart",
     "question": "According to Figure 2.4, which country shows the highest growth rate in IT services exports, 2022-23?",
     "gold_answer": "Cambodia (KHM)"},
    {"doc_name": "AIFoundations", "page": 85, "modality": "chart",
     "question": "According to Figure 4.3 panel c, what percentage of open-source data sets from Hugging Face are in English?",
     "gold_answer": "56.3 percent"},
    {"doc_name": "AIFoundations", "page": 95, "modality": "diagram",
     "question": "According to Figure 5.1, what basic-level digital-skill activities are listed under 'Consumers'?",
     "gold_answer": "Simple smartphone usage, basic internet, social media, digital payments"},
    {"doc_name": "AIFoundations", "page": 104, "modality": "chart",
     "question": "According to Figure 5.5 panel b, which country-income-group and skill-level combination shows the largest urban-rural gap in digital skills?",
     "gold_answer": "LMICs at the Advanced skill level (about 5.2 percentage points)"},
    {"doc_name": "AIFoundations", "page": 108, "modality": "chart",
     "question": "According to Figure 5.8 panel a, which country has the highest ratio of high-skilled talent outflow to inflow?",
     "gold_answer": "Ukraine (3.9)"},
    {"doc_name": "AIFoundations", "page": 115, "modality": "table",
     "question": "According to Table 6.1, what is recommended for 'Competency' in countries with High AI readiness?",
     "gold_answer": "Develop advanced digital and AI skills; develop and support top-notch AI researchers; attract and retain talent"},
    {"doc_name": "AIFoundations", "page": 117, "modality": "chart",
     "question": "According to Figure 6.1, which country has the largest bubble (highest AI readiness scale value)?",
     "gold_answer": "United States"},
]

print(f"Total questions: {len(QA_SET)}")
from collections import Counter
print("Per document:", Counter(q["doc_name"] for q in QA_SET))
print("Per modality:", Counter(q["modality"] for q in QA_SET))


Total questions: 30
Per document: Counter({'2407_01449v6': 10, 'WorldDevelopmentReport': 10, 'AIFoundations': 10})
Per modality: Counter({'chart': 15, 'table': 7, 'text': 3, 'map': 2, 'diagram': 2, 'infographic': 1})


In [8]:
for q in QA_SET:
    q["doc_name"] = q["doc_name"].replace("2407_01449v6", "2407.01449v6")
print(sorted({q["doc_name"] for q in QA_SET}))

['2407.01449v6', 'AIFoundations', 'WorldDevelopmentReport']


## 5b. Benchmark balance -- optional extra text-heavy questions

Only 3 of the 30 questions are text-type and 15 are charts. That is fine for *this* evaluation, but Evaluation 2 is meant to show **where visual retrieval wins**, and a chart-heavy question set would look rigged to a reviewer. Adding roughly 10 hand-verified, page-grounded **text-heavy** questions lets Evaluation 2 also show where OCR holds up or wins, which makes any visual advantage more credible.

This cell is deliberately **empty by default** -- gold pages have to be hand-verified against the actual PDFs, so they can't be generated blindly. Fill `QA_SET_EXTRA` using the same schema as `QA_SET` (`doc_name`, `page`, `modality`, `question`, `gold_answer`).

Because the generation loop is checkpointed per question index, appending questions is cheap: re-run the Section 6 retrieval loop and the Section 7 generation cell, and **only the new questions call Gemini**. Do this *before* running Sections 6b-8b so every number and the frozen benchmark include them.

In [9]:
from collections import Counter

# Optional: add hand-verified, page-grounded questions here (same schema as QA_SET).
# Leave the list empty to keep the benchmark at 30 questions.
QA_SET_EXTRA = [
    # {"doc_name": "AIFoundations", "page": 0, "modality": "text",
    #  "question": "...", "gold_answer": "..."},
]

_page_exists = {(c["doc_name"], c["page_number"]) for c in CORPUS}
_known = {(q["doc_name"], q["page"], q["question"]) for q in QA_SET}
_added = 0
for _q in QA_SET_EXTRA:
    _q = dict(_q)
    _q["doc_name"] = _q["doc_name"].replace("2407_01449v6", "2407.01449v6")
    missing = {"doc_name", "page", "modality", "question", "gold_answer"} - set(_q)
    if missing:
        raise ValueError(f"Extra question is missing fields {missing}: {_q}")
    if (_q["doc_name"], _q["page"]) not in _page_exists:
        raise ValueError(f"No indexed page for {_q['doc_name']} p.{_q['page']} -- check doc_name / page number.")
    key = (_q["doc_name"], _q["page"], _q["question"])
    if key not in _known:
        QA_SET.append(_q)
        _known.add(key)
        _added += 1

print(f"Added {_added} extra question(s). Benchmark size now: {len(QA_SET)}")
print("Per modality:", Counter(q["modality"] for q in QA_SET))
if _added:
    print("\nRe-run Section 6 (retrieval loop) and the Section 7 generation cell.")
    print("The generation checkpoint means only the NEW questions call Gemini.")

Added 0 extra question(s). Benchmark size now: 30
Per modality: Counter({'chart': 15, 'table': 7, 'text': 3, 'map': 2, 'diagram': 2, 'infographic': 1})


## 6. Retrieval evaluation -- Recall@1 / @3 / @5 / @10

For each question, run `exhaustive_maxsim_search` with `top_k=10` once, then check whether the gold `(doc_name, page)` pair appears within the top-1 / top-3 / top-5 / top-10 of that single ranked list (cheaper than re-querying per k, and equivalent).

In [10]:
import ast, math

# Set to False to force a fresh retrieval run (about 8 minutes on CPU, ~16 s/query).
USE_SAVED_RETRIEVAL = True
_SAVED_CSV = "/content/drive/MyDrive/Visual-Doc/eval_results/eval1_retrieval_results.csv"

retrieval_results = None
if USE_SAVED_RETRIEVAL and os.path.exists(_SAVED_CSV):
    _s = pd.read_csv(_SAVED_CSV) if "pd" in globals() else __import__("pandas").read_csv(_SAVED_CSV)
    _ok = (len(_s) == len(QA_SET)
           and all(str(_s.loc[i, "question"]) == QA_SET[i]["question"] for i in range(len(QA_SET)))
           and all(_s.loc[i, "doc_name"] == QA_SET[i]["doc_name"] and int(_s.loc[i, "gold_page"]) == QA_SET[i]["page"]
                   for i in range(len(QA_SET))))
    if _ok:
        retrieval_results = []
        for _, r in _s.iterrows():
            rk = None if (isinstance(r["rank_of_gold"], float) and math.isnan(r["rank_of_gold"])) else int(r["rank_of_gold"])
            retrieval_results.append({
                "idx": int(r["idx"]), "doc_name": r["doc_name"], "gold_page": int(r["gold_page"]),
                "modality": r["modality"], "question": r["question"], "rank_of_gold": rk,
                "top1_doc": r["top1_doc"], "top1_page": int(r["top1_page"]),
                "top10": [tuple(t) for t in ast.literal_eval(r["top10"])],
                "hit_at_1": bool(r["hit_at_1"]), "hit_at_3": bool(r["hit_at_3"]),
                "hit_at_5": bool(r["hit_at_5"]), "hit_at_10": bool(r["hit_at_10"]),
            })
        print(f"Loaded {len(retrieval_results)} saved retrieval results from {_SAVED_CSV}")
        print("(Original run: exhaustive MaxSim, CPU. Set USE_SAVED_RETRIEVAL = False to recompute.)")
    else:
        print("Saved retrieval CSV does not match the current QA_SET -- recomputing.")

if retrieval_results is None:
    retrieval_results = []

if not retrieval_results:
    for i, qa in enumerate(QA_SET):
        ranked = exhaustive_maxsim_search(qa["question"], top_k=10)
        gold = (qa["doc_name"], qa["page"])
        rank = None
        for idx, hit in enumerate(ranked):
            if (hit["doc_name"], hit["page_number"]) == gold:
                rank = idx + 1
                break

        retrieval_results.append({
            "idx": i,
            "doc_name": qa["doc_name"],
            "gold_page": qa["page"],
            "modality": qa["modality"],
            "question": qa["question"],
            "rank_of_gold": rank,
            "top1_doc": ranked[0]["doc_name"],
            "top1_page": ranked[0]["page_number"],
            "top10": [(h["doc_name"], h["page_number"]) for h in ranked],
            "hit_at_1": rank == 1,
            "hit_at_3": rank is not None and rank <= 3,
            "hit_at_5": rank is not None and rank <= 5,
            "hit_at_10": rank is not None and rank <= 10,
        })
        status = f"rank {rank}" if rank else "NOT in top 10"
        print(f"[{i+1:>2}/{len(QA_SET)}] {qa['doc_name']:<24} p.{qa['page']:<4} ({qa['modality']:<11}) -> {status}")

Loaded 30 saved retrieval results from /content/drive/MyDrive/Visual-Doc/eval_results/eval1_retrieval_results.csv
(Original run: exhaustive MaxSim, CPU. Set USE_SAVED_RETRIEVAL = False to recompute.)


### Aggregate Recall@k -- overall, per document, per modality

In [11]:
import pandas as pd

df = pd.DataFrame(retrieval_results)

def recall_table(frame, group_col=None):
    if group_col is None:
        row = {
            "n": len(frame),
            "Recall@1": frame["hit_at_1"].mean(),
            "Recall@3": frame["hit_at_3"].mean(),
            "Recall@5": frame["hit_at_5"].mean(),
            "Recall@10": frame["hit_at_10"].mean(),
        }
        return pd.DataFrame([row], index=["ALL"])
    return frame.groupby(group_col).agg(
        n=("hit_at_1", "size"),
        **{"Recall@1": ("hit_at_1", "mean"),
           "Recall@3": ("hit_at_3", "mean"),
           "Recall@5": ("hit_at_5", "mean"),
           "Recall@10": ("hit_at_10", "mean")}
    )

print("=== Overall ===")
display(recall_table(df))

print()
print("=== By document ===")
display(recall_table(df, "doc_name"))

print()
print("=== By modality ===")
display(recall_table(df, "modality"))

print()
print("=== Misses (gold page not in top 10 -- worth inspecting) ===")
misses = df[df["rank_of_gold"].isna()]
if len(misses):
    display(misses[["doc_name", "gold_page", "modality", "question", "top1_doc", "top1_page"]])
else:
    print("None -- every gold page was retrieved somewhere in the top 10.")


=== Overall ===


,n,Recall@1,Recall@3,Recall@5,Recall@10
ALL,30,0.866667,0.966667,0.966667,1.0



=== By document ===


,n,Recall@1,Recall@3,Recall@5,Recall@10
doc_name,,,,,
2407.01449v6,10,0.7,0.9,0.9,1.0
AIFoundations,10,0.9,1.0,1.0,1.0
WorldDevelopmentReport,10,1.0,1.0,1.0,1.0



=== By modality ===


,n,Recall@1,Recall@3,Recall@5,Recall@10
modality,,,,,
chart,15,0.866667,1.000000,1.000000,1.0
diagram,2,1.000000,1.000000,1.000000,1.0
infographic,1,1.000000,1.000000,1.000000,1.0
map,2,1.000000,1.000000,1.000000,1.0
table,7,0.857143,1.000000,1.000000,1.0
text,3,0.666667,0.666667,0.666667,1.0



=== Misses (gold page not in top 10 -- worth inspecting) ===
None -- every gold page was retrieved somewhere in the top 10.


In [12]:
print(sum(r["hit_at_1"] for r in retrieval_results), "/ 30")

26 / 30


## 6b. Full ranking metrics -- MRR@10 and nDCG@5, via a shared metric module

Section 6 reports Recall@k only; the evaluation plan also promises **MRR** and **nDCG@5**. Each question has exactly one gold page, so MRR@10 is the mean of `1/rank` (0 if outside the top 10) and nDCG@5 is `1/log2(rank+1)` when the gold page is in the top 5 (the ideal DCG is 1).

The metric code is written to `eval_metrics.py` on Drive instead of being defined inline. **Evaluation 2 imports the same file**, so the OCR arm and the ColPali arm are scored by identical code and any gap in the numbers comes from retrieval, not from two slightly different metric implementations. The file also holds the answer-scoring helpers used in Section 7b.

In [13]:
%%writefile /content/drive/MyDrive/Visual-Doc/eval_metrics.py
"""
Visual-Doc evaluation helpers -- shared by Evaluation 1 and Evaluation 2.

Both retrieval arms (ColPali + MaxSim now, OCR/text RAG later) must be scored by
IDENTICAL code, otherwise a difference in the numbers could come from the
metric implementation instead of the retrieval method. Everything that turns a
ranked list into a number, or a model answer into a verdict, lives here.

Retrieval convention: every question has exactly ONE gold page, identified as
(doc_name, page_number). `rank_of_gold` is its 1-based position in the ranked
list (None if it is not in the top max_k).
"""
import math
import re

import pandas as pd

KS = (1, 3, 5, 10)
MAX_K = 10


# --------------------------------------------------------------------------
# Retrieval metrics
# --------------------------------------------------------------------------
def rank_of_gold(ranked_pairs, gold, max_k=MAX_K):
    """1-based rank of `gold` within the first max_k (doc_name, page) pairs, else None."""
    for i, pair in enumerate(list(ranked_pairs)[:max_k]):
        if tuple(pair) == tuple(gold):
            return i + 1
    return None


def _clean_rank(r):
    if r is None:
        return None
    if isinstance(r, float) and math.isnan(r):
        return None
    return int(r)


def hit_at_k(rank, k):
    rank = _clean_rank(rank)
    return rank is not None and rank <= k


def reciprocal_rank(rank, max_k=MAX_K):
    rank = _clean_rank(rank)
    return 1.0 / rank if rank is not None and rank <= max_k else 0.0


def ndcg_at_k(rank, k=5):
    """nDCG@k with ONE relevant page: DCG = 1/log2(rank+1) if rank<=k, IDCG = 1."""
    rank = _clean_rank(rank)
    return 1.0 / math.log2(rank + 1) if rank is not None and rank <= k else 0.0


def metrics_from_ranks(ranks):
    ranks = [_clean_rank(r) for r in ranks]
    n = len(ranks)
    out = {"n": n}
    for k in KS:
        out[f"Recall@{k}"] = sum(hit_at_k(r, k) for r in ranks) / n if n else float("nan")
    out["MRR@10"] = sum(reciprocal_rank(r) for r in ranks) / n if n else float("nan")
    out["nDCG@5"] = sum(ndcg_at_k(r, 5) for r in ranks) / n if n else float("nan")
    return out


def metrics_table(frame, group_col=None, rank_col="rank_of_gold"):
    """Recall@1/3/5/10, MRR@10 and nDCG@5, overall (group_col=None) or per group."""
    if group_col is None:
        return pd.DataFrame([metrics_from_ranks(frame[rank_col].tolist())], index=["ALL"])
    rows = {g: metrics_from_ranks(sub[rank_col].tolist()) for g, sub in frame.groupby(group_col)}
    return pd.DataFrame(rows).T.astype({"n": int})


# --------------------------------------------------------------------------
# Answer scoring
# --------------------------------------------------------------------------
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9.% ]", " ", text)
    text = re.sub(r" +", " ", text)
    return text.strip()


def extract_keyphrases(gold_answer):
    """Verbatim copy of the Evaluation 1 notebook's helper (kept identical on purpose)."""
    numbers = re.findall(r"\d+(?:\.\d+)?%?", gold_answer)
    proper_nouns = re.findall(r"\b[A-Z][a-zA-Z-]+\b", gold_answer)
    seen = []
    for kp in numbers + proper_nouns:
        if kp not in seen:
            seen.append(kp)
    return seen if seen else [gold_answer]


def keyword_grounded_score(model_answer, gold_answer):
    """Verbatim copy of the ORIGINAL Evaluation 1 scorer (lenient: needs half the key phrases)."""
    norm_answer = normalize(model_answer)
    keyphrases = extract_keyphrases(gold_answer)
    hits = 0
    for kp in keyphrases:
        if normalize(kp) in norm_answer:
            hits += 1
    return hits >= max(1, len(keyphrases) // 2)


# Phrases a model uses when it says the page does not contain the answer.
ABSTAIN_PATTERNS = [
    r"\bnot (?:included|contained|present|shown|visible|available|provided|found|mentioned|listed|displayed)\b",
    r"\b(?:does|do|did)(?: not|n't) (?:contain|include|show|provide|mention|list|have|display)\b",
    r"\b(?:cannot|can't|can not|could not|couldn't|unable to) (?:find|answer|be answered|determine|locate|identify)\b",
    r"\bno (?:information|mention|data|reference)\b",
    r"\bnot (?:enough|sufficient) information\b",
    r"\bnot (?:on|in) (?:this|the|its|the provided|the given) (?:page|image|document)\b",
    r"\bis not present\b|\bare not present\b",
]


def is_abstention(model_answer):
    """True if the answer says the provided page(s) do not contain the answer."""
    a = model_answer.lower()
    return any(re.search(p, a) for p in ABSTAIN_PATTERNS)


def numbers_all_present(model_answer, gold_answer):
    """Every number in the gold answer must appear (as a whole number) in the answer."""
    nums = re.findall(r"\d+(?:\.\d+)?", gold_answer)
    if not nums:
        return True
    norm = normalize(model_answer)
    for n in nums:
        if not re.search(r"(?<![\d.])" + re.escape(n) + r"(?!\d|\.\d)", norm):
            return False
    return True


def strict_keyword_score(model_answer, gold_answer):
    """Provisional automatic verdict: NOT an abstention, ALL gold numbers present,
    and the original keyword check also passes. Tighter than keyword_grounded_score
    (which could pass an answer like 'Table 2 is not included on this page' because
    generic words such as 'document' or 'retrieval' matched)."""
    return (
        (not is_abstention(model_answer))
        and numbers_all_present(model_answer, gold_answer)
        and keyword_grounded_score(model_answer, gold_answer)
    )

Overwriting /content/drive/MyDrive/Visual-Doc/eval_metrics.py


In [14]:
import sys, importlib

EVAL_DIR = "/content/drive/MyDrive/Visual-Doc"
if EVAL_DIR not in sys.path:
    sys.path.append(EVAL_DIR)

import eval_metrics
importlib.reload(eval_metrics)
from eval_metrics import (metrics_table, metrics_from_ranks, rank_of_gold,
                          is_abstention, strict_keyword_score, numbers_all_present)

print("=== Overall (Recall@k, MRR@10, nDCG@5) ===")
display(metrics_table(df))

print()
print("=== By document ===")
display(metrics_table(df, "doc_name"))

print()
print("=== By modality ===")
display(metrics_table(df, "modality"))

# Consistency check: the shared module must reproduce the Recall numbers from Section 6.
_old = recall_table(df).iloc[0]
_new = metrics_table(df).iloc[0]
_same = all(abs(_old[f"Recall@{k}"] - _new[f"Recall@{k}"]) < 1e-12 for k in (1, 3, 5, 10))
print()
print("Shared metric module reproduces the Section 6 Recall@k numbers:", _same)

=== Overall (Recall@k, MRR@10, nDCG@5) ===


,n,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@5
ALL,30,0.866667,0.966667,0.966667,1.0,0.910317,0.921031



=== By document ===


,n,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@5
2407.01449v6,10,0.7,0.9,0.9,1.0,0.780952,0.800000
AIFoundations,10,0.9,1.0,1.0,1.0,0.950000,0.963093
WorldDevelopmentReport,10,1.0,1.0,1.0,1.0,1.000000,1.000000



=== By modality ===


,n,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@5
chart,15,0.866667,1.000000,1.000000,1.0,0.922222,0.942062
diagram,2,1.000000,1.000000,1.000000,1.0,1.000000,1.000000
infographic,1,1.000000,1.000000,1.000000,1.0,1.000000,1.000000
map,2,1.000000,1.000000,1.000000,1.0,1.000000,1.000000
table,7,0.857143,1.000000,1.000000,1.0,0.904762,0.928571
text,3,0.666667,0.666667,0.666667,1.0,0.714286,0.666667



Shared metric module reproduces the Section 6 Recall@k numbers: True


## 6c. Efficiency -- query latency, storage per page, indexing time per page

Efficiency is where OCR is most likely to win, so the visual arm's costs have to be measured, not just its accuracy.

**This cell runs on a plain CPU runtime (no T4 needed).** What it measures, and how comparable each number is:

| Metric | Runs on | Comparable? |
|---|---|---|
| Storage per page (patch matrix, proxy, PNG, ChromaDB folder) | disk | Yes, device independent |
| Exhaustive MaxSim time over all pages | NumPy on the CPU | Yes, same on any runtime |
| Query embedding time | the model's device | **No**: recorded with the device name; CPU is much slower than the T4 |
| Indexing time per page | the model's device | **Skipped on CPU** (set `MEASURE_INDEXING = True` to force a small sample) |

The device is saved with every number. In the report, quote storage and MaxSim time as measured, quote the query-embedding time as "CPU", and for indexing time cite the ColPali paper's figure (0.39 s/page) as a reference, stating that it was not measured here.

It also caches every question's query embedding (`QUERY_EMB`, saved to Drive) for Section 6d.

In [15]:
import time, random, tempfile, gc, pickle

def _sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none (CPU)"
ON_GPU = DEVICE == "cuda"
MEASURE_INDEXING = ON_GPU      # False on CPU by default; set True to time a small sample anyway
print(f"Device: {DEVICE}  |  GPU: {GPU_NAME}")
if not ON_GPU:
    print("CPU runtime: query-embedding time is labelled CPU; indexing time is skipped (see the table above).")

QUERY_CACHE = "/content/drive/MyDrive/Visual-Doc/eval_results/query_emb_cache.pkl"

# ------------------------------------------------------------------ query latency
_ = embed_query("warm-up query"); _sync()

QUERY_EMB = {}
t_embed, t_maxsim, n_qtok = [], [], []
for i, qa in enumerate(QA_SET):
    t0 = time.perf_counter()
    proxy, qtok = embed_query(qa["question"]); _sync()
    t1 = time.perf_counter()
    scores = [maxsim_score(qtok, p["patches"]) for p in CORPUS]       # exhaustive MaxSim (NumPy, CPU)
    _top = sorted(range(len(scores)), key=lambda j: scores[j], reverse=True)[:10]
    t2 = time.perf_counter()
    QUERY_EMB[i] = (proxy, qtok)
    t_embed.append(t1 - t0); t_maxsim.append(t2 - t1); n_qtok.append(qtok.shape[0])
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(QA_SET)} queries timed")

os.makedirs(os.path.dirname(QUERY_CACHE), exist_ok=True)
with open(QUERY_CACHE, "wb") as f:
    pickle.dump({"questions": [q["question"] for q in QA_SET], "emb": QUERY_EMB, "device": DEVICE}, f)

def _ms(xs):
    a = np.array(xs) * 1000
    return {"mean_ms": float(a.mean()), "median_ms": float(np.median(a)), "p95_ms": float(np.percentile(a, 95))}

lat_embed, lat_maxsim = _ms(t_embed), _ms(t_maxsim)
lat_total = _ms([a + b for a, b in zip(t_embed, t_maxsim)])
print(f"\nQuery latency over {len(QA_SET)} questions, {len(CORPUS)} pages (retrieval only):")
print(f"  query embedding [{DEVICE}] : mean {lat_embed['mean_ms']:.0f} ms | median {lat_embed['median_ms']:.0f} | p95 {lat_embed['p95_ms']:.0f}")
print(f"  MaxSim over all [CPU numpy]: mean {lat_maxsim['mean_ms']:.0f} ms | median {lat_maxsim['median_ms']:.0f} | p95 {lat_maxsim['p95_ms']:.0f}")
print(f"  total retrieval [{DEVICE}]  : mean {lat_total['mean_ms']:.0f} ms | median {lat_total['median_ms']:.0f} | p95 {lat_total['p95_ms']:.0f}")
print(f"  query tokens               : mean {np.mean(n_qtok):.1f} (incl. ColBERT-style augmentation tokens)")

# ------------------------------------------------------------------ storage / page
metas = all_data["metadatas"]
patch_bytes = [os.path.getsize(m["patch_embedding_path"]) for m in metas
               if m.get("patch_embedding_path") and os.path.exists(m["patch_embedding_path"])]
png_bytes = [os.path.getsize(m["image_path"]) for m in metas if os.path.exists(m["image_path"])]
n_patches = [m["num_patches"] for m in metas if "num_patches" in m] or [int(b / (128 * 2)) for b in patch_bytes]
PROXY_BYTES = 128 * 4

def _dir_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        total += sum(os.path.getsize(os.path.join(root, f)) for f in files)
    return total / 1e6

storage = {
    "patch_matrix_kb_per_page": float(np.mean(patch_bytes) / 1024),
    "mean_patches_per_page": float(np.mean(n_patches)),
    "proxy_vector_bytes_per_page": PROXY_BYTES,
    "page_png_kb_per_page": float(np.mean(png_bytes) / 1024) if png_bytes else None,
    "patch_store_total_mb": float(sum(patch_bytes) / 1e6),
    "chromadb_dir_mb": _dir_mb(DB_PATH),
    "in_memory_corpus_float32_mb": float(total_mb),
}
print("\nStorage per page (float16 patch matrices on disk):")
print(f"  full patch matrix : {storage['patch_matrix_kb_per_page']:.1f} KB  (~{storage['mean_patches_per_page']:.0f} patches x 128 dims)")
print(f"  pooled proxy      : {PROXY_BYTES} B  (stored in ChromaDB)")
if storage["page_png_kb_per_page"]:
    print(f"  page image (PNG)  : {storage['page_png_kb_per_page']:.0f} KB")
print(f"  patch store total : {storage['patch_store_total_mb']:.1f} MB for {len(patch_bytes)} pages")
print(f"  ChromaDB folder   : {storage['chromadb_dir_mb']:.1f} MB")
print(f"  in RAM (float32)  : {storage['in_memory_corpus_float32_mb']:.1f} MB")

# ------------------------------------------------------------------ indexing time / page
embed_s_per_page = save_s_per_page = render_s_per_page = index_total = batch_used = None
SAMPLE_PAGES = 8 if ON_GPU else 2
EMBED_BATCH, RENDER_DPI = 4, 200

if not MEASURE_INDEXING:
    print("\nIndexing time: NOT measured (CPU runtime). Report it as 'not measured' and cite the ColPali")
    print("paper's 0.39 s/page (Fig. 1, paper hardware) as a reference only.")
else:
    gc.collect()
    if ON_GPU:
        torch.cuda.empty_cache()
    _rng = random.Random(0)
    _sample = _rng.sample(CORPUS, SAMPLE_PAGES)
    _imgs = [Image.open(p["image_path"]).convert("RGB") for p in _sample]

    def _time_indexing(batch):
        embed_page_images_batch(_imgs[:1]); _sync()
        t0 = time.perf_counter()
        out = []
        for b in range(0, len(_imgs), batch):
            out += embed_page_images_batch(_imgs[b:b + batch])
        _sync()
        return (time.perf_counter() - t0) / len(_imgs), out

    OOM = torch.cuda.OutOfMemoryError if ON_GPU else MemoryError
    _embedded = []
    for _batch in (EMBED_BATCH, 1):
        try:
            embed_s_per_page, _embedded = _time_indexing(_batch)
            batch_used = _batch
            break
        except OOM:
            gc.collect()
            if ON_GPU:
                torch.cuda.empty_cache()
            print(f"Out of memory at batch size {_batch}" + (" -- retrying with batch size 1" if _batch != 1 else ""))
    if embed_s_per_page is not None:
        with tempfile.TemporaryDirectory() as _tmp:
            t0 = time.perf_counter()
            for k, (full, _proxy) in enumerate(_embedded):
                np.save(os.path.join(_tmp, f"p{k}.npy"), full)
            save_s_per_page = (time.perf_counter() - t0) / len(_embedded)
        index_total = embed_s_per_page + save_s_per_page
        print(f"\nIndexing time per page ({SAMPLE_PAGES} sampled pages, batch={batch_used}, device={DEVICE}):")
        print(f"  ColPali embedding : {embed_s_per_page:.3f} s/page")
        print(f"  patch-matrix save : {save_s_per_page:.4f} s/page")
        print(f"  total (excl. rendering): {index_total:.3f} s/page")

EFFICIENCY = {
    "device": DEVICE, "gpu": GPU_NAME,
    "query_latency_ms": {"embedding": lat_embed, "maxsim_exhaustive": lat_maxsim, "total_retrieval": lat_total},
    "latency_notes": {"embedding_device": DEVICE, "maxsim_runs_on": "CPU (NumPy float32)",
                      "comparable_across_runtimes": ["maxsim_exhaustive", "storage"],
                      "device_dependent": ["embedding", "total_retrieval", "indexing_s_per_page"]},
    "mean_query_tokens": float(np.mean(n_qtok)),
    "storage": storage,
    "indexing_s_per_page": {"embedding": embed_s_per_page, "patch_save": save_s_per_page,
                            "rendering": render_s_per_page, "total": index_total,
                            "measured": index_total is not None,
                            "sample_pages": SAMPLE_PAGES if index_total is not None else 0,
                            "batch": batch_used, "render_dpi": RENDER_DPI,
                            "reference_paper_s_per_page": 0.39},
}


Device: cpu  |  GPU: none (CPU)
CPU runtime: query-embedding time is labelled CPU; indexing time is skipped (see the table above).
  10/30 queries timed
  20/30 queries timed
  30/30 queries timed

Query latency over 30 questions, 521 pages (retrieval only):
  query embedding [cpu] : mean 17152 ms | median 16721 | p95 21348
  MaxSim over all [CPU numpy]: mean 344 ms | median 216 | p95 1139
  total retrieval [cpu]  : mean 17496 ms | median 17185 | p95 21564
  query tokens               : mean 39.1 (incl. ColBERT-style augmentation tokens)

Storage per page (float16 patch matrices on disk):
  full patch matrix : 257.9 KB  (~1031 patches x 128 dims)
  pooled proxy      : 512 B  (stored in ChromaDB)
  page image (PNG)  : 585 KB
  patch store total : 137.6 MB for 521 pages
  ChromaDB folder   : 1.9 MB
  in RAM (float32)  : 275.0 MB

Indexing time: NOT measured (CPU runtime). Report it as 'not measured' and cite the ColPali
paper's 0.39 s/page (Fig. 1, paper hardware) as a reference only.


## 6d. Retrieval-mode ablation -- mean-pooled cosine vs. two-stage vs. exhaustive MaxSim

The evaluation plan lists this ablation as part of the system evaluation. It compares the three retrieval configurations this project went through, on the same 30 questions and the same 521 pages:

| Mode | What it does |
|---|---|
| Mean-pooled cosine | Steps 9/9B: ChromaDB cosine search over one mean-pooled vector per page, no MaxSim |
| Two-stage MaxSim | Step 9D "Fast": ChromaDB shortlist (20 or 50), then exact MaxSim re-rank of the shortlist |
| Exhaustive MaxSim | Every page scored exactly -- the **headline** configuration used in Sections 6-7 |

"Shortlist recall" is how often the gold page even makes it into the two-stage shortlist; when it doesn't, MaxSim re-ranking can't rescue it.

**Scope note:** this is the *only* section that re-implements the mean-pooled and two-stage paths, in isolated functions, purely to compare them. Nothing in Sections 6-7 touches them, so the headline numbers remain exhaustive-MaxSim-only. A consistency check at the end confirms the exhaustive row here reproduces the Section 6 ranks.

**CPU-only:** every mode here runs on the CPU. The query embeddings come from the Section 6c cache (`QUERY_EMB`) or the cache file on Drive, and are only recomputed if neither exists (30 queries, a few minutes on CPU).

In [16]:
# ISOLATED section: re-implements the mean-pooled and two-stage paths purely to compare
# them with exhaustive MaxSim. Nothing in Sections 6-7 uses these functions.
import pickle
_QC = "/content/drive/MyDrive/Visual-Doc/eval_results/query_emb_cache.pkl"
if "QUERY_EMB" not in globals():
    QUERY_EMB = None
    if os.path.exists(_QC):
        with open(_QC, "rb") as _f:
            _d = pickle.load(_f)
        if _d["questions"] == [q["question"] for q in QA_SET]:
            QUERY_EMB = _d["emb"]
            print(f"Loaded cached query embeddings (computed on {_d['device']}).")
    if QUERY_EMB is None:
        print("No usable cache -- embedding the queries now (a few minutes on CPU)...")
        QUERY_EMB = {i: embed_query(q["question"]) for i, q in enumerate(QA_SET)}

CORPUS_INDEX = {(c["doc_name"], c["page_number"]): c for c in CORPUS}

def _chroma_candidates(proxy, n):
    res = collection.query(query_embeddings=[proxy.tolist()], n_results=n, include=["metadatas"])
    return [(m["doc_name"], m["page_number"]) for m in res["metadatas"][0]]

def search_mean_pooled(proxy, top_k=10):
    """Steps 9/9B: ChromaDB cosine search over the mean-pooled page vector, no MaxSim."""
    return _chroma_candidates(proxy, top_k), None

def search_two_stage(proxy, qtok, shortlist=20, top_k=10):
    """Step 9D 'Fast' mode: ChromaDB shortlist, then exact MaxSim re-rank of the shortlist."""
    cands = _chroma_candidates(proxy, shortlist)
    scored = sorted(((maxsim_score(qtok, CORPUS_INDEX[k]["patches"]), k) for k in cands if k in CORPUS_INDEX),
                    reverse=True)
    return [k for _, k in scored[:top_k]], cands

def search_exhaustive(qtok, top_k=10):
    scored = sorted(((maxsim_score(qtok, c["patches"]), (c["doc_name"], c["page_number"])) for c in CORPUS),
                    reverse=True)
    return [k for _, k in scored[:top_k]], None

MODES = {
    "Mean-pooled cosine (Steps 9/9B)":          lambda p, q: search_mean_pooled(p),
    "Two-stage MaxSim, shortlist 20 (app default)": lambda p, q: search_two_stage(p, q, 20),
    "Two-stage MaxSim, shortlist 50":           lambda p, q: search_two_stage(p, q, 50),
    "Exhaustive MaxSim (headline)":             lambda p, q: search_exhaustive(q),
}

abl_ranks = {name: [] for name in MODES}
abl_ms = {name: [] for name in MODES}
abl_top1 = {name: [] for name in MODES}
shortlist_hit = {name: [] for name in MODES}

for i, qa in enumerate(QA_SET):
    proxy, qtok = QUERY_EMB[i]
    gold = (qa["doc_name"], qa["page"])
    for name, fn in MODES.items():
        t0 = time.perf_counter()
        ranked, cands = fn(proxy, qtok)
        abl_ms[name].append((time.perf_counter() - t0) * 1000)
        abl_ranks[name].append(rank_of_gold(ranked, gold))
        abl_top1[name].append(ranked[0] if ranked else None)
        if cands is not None:
            shortlist_hit[name].append(gold in cands)

exh = "Exhaustive MaxSim (headline)"
rows = {}
for name in MODES:
    r = metrics_from_ranks(abl_ranks[name])
    r["Shortlist recall"] = float(np.mean(shortlist_hit[name])) if shortlist_hit[name] else np.nan
    r["Top-1 agrees w/ exhaustive"] = float(np.mean([a == b for a, b in zip(abl_top1[name], abl_top1[exh])]))
    r["Search ms/query"] = float(np.mean(abl_ms[name]))
    rows[name] = r
ABLATION = pd.DataFrame(rows).T.drop(columns="n")
print(f"Retrieval-mode ablation -- {len(QA_SET)} questions, {len(CORPUS)} pages, search stage only "
      f"(query embedding excluded; it is identical across modes):\n")
display(ABLATION.round(3))

# Consistency: the exhaustive row must match the Section 6 ranks.
_sec6 = [r["rank_of_gold"] for r in retrieval_results[:len(QA_SET)]]
print("\nExhaustive ranks here == Section 6 ranks:", abl_ranks[exh] == _sec6)

# Questions where the modes disagree about where the gold page lands.
cmp = pd.DataFrame({name: abl_ranks[name] for name in MODES})
cmp.insert(0, "question", [q["question"][:70] for q in QA_SET])
cmp.insert(0, "doc", [q["doc_name"] for q in QA_SET])
cmp.insert(1, "page", [q["page"] for q in QA_SET])
differ = cmp[cmp[list(MODES)].nunique(axis=1, dropna=False) > 1]
print(f"\nQuestions where at least one mode ranks the gold page differently ({len(differ)} of {len(cmp)}); "
      f"NaN = not in top 10:")
display(differ)

ABLATION_PER_QUESTION = cmp

Retrieval-mode ablation -- 30 questions, 521 pages, search stage only (query embedding excluded; it is identical across modes):



,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@5,Shortlist recall,Top-1 agrees w/ exhaustive,Search ms/query
Mean-pooled cosine (Steps 9/9B),0.367,0.600,0.700,0.967,0.527,0.542,NaN,0.400,6.199
"Two-stage MaxSim, shortlist 20 (app default)",0.833,0.933,0.933,0.967,0.878,0.888,0.967,0.967,20.197
"Two-stage MaxSim, shortlist 50",0.833,0.933,0.933,0.967,0.877,0.888,0.967,0.967,53.096
Exhaustive MaxSim (headline),0.867,0.967,0.967,1.000,0.910,0.921,NaN,1.000,319.808



Exhaustive ranks here == Section 6 ranks: True

Questions where at least one mode ranks the gold page differently (19 of 30); NaN = not in top 10:


,doc,page,question,Mean-pooled cosine (Steps 9/9B),"Two-stage MaxSim, shortlist 20 (app default)","Two-stage MaxSim, shortlist 50",Exhaustive MaxSim (headline)
0,2407.01449v6,1,What is the name of the benchmark introduced i...,5.0,6.0,7.0,7
1,2407.01449v6,2,"According to Figure 1, what is the offline ind...",2.0,1.0,1.0,1
2,2407.01449v6,4,"According to Table 1, how many queries and how...",2.0,1.0,1.0,1
3,2407.01449v6,5,"According to Figure 2, what is the total per-p...",6.0,3.0,3.0,3
4,2407.01449v6,7,"According to Table 2, what nDCG@5 score does C...",4.0,3.0,3.0,3
7,2407.01449v6,19,"According to Table 5, how long does the layout...",8.0,1.0,1.0,1
12,WorldDevelopmentReport,83,"According to Figure 1.7, in developing countri...",3.0,1.0,1.0,1
14,WorldDevelopmentReport,132,"According to Table 2.2, what is the potential-...",7.0,1.0,1.0,1
15,WorldDevelopmentReport,155,"According to Figure 2.24, which country has th...",6.0,1.0,1.0,1
17,WorldDevelopmentReport,227,"According to Figure B4.1.1, how many billion p...",2.0,1.0,1.0,1


## 7. Generation evaluation

Given the page(s) retrieval actually returned, does Gemini produce the correct answer? This reuses your `generate_answer` / `build_multimodal_input` / `SYSTEM_INSTRUCTION` from Step 9D so the generation step is identical to what the deployed app does -- only the retrieval call feeding it has been swapped for the exhaustive in-memory version above.

**On scoring accuracy:** free-text answers can't be string-matched exactly, so this uses a lightweight *keyword-grounded* scorer -- it checks whether the key fact(s) from `gold_answer` (numbers, names) appear in Gemini's response, normalized for case/punctuation. This is transparent and good enough to catch clear right/wrong answers, but it's not a substitute for human review or an LLM-judge for partial-credit/faithfulness scoring. A stub for swapping in an LLM-judge is included at the end of this section if you want a more nuanced metric later.

In [17]:
import os
from getpass import getpass
from google import genai
import base64

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key (from https://aistudio.google.com/apikey): ")

gemini_client = genai.Client()
GEMINI_MODEL = "gemini-3.6-flash"

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below -- "
    "do not use outside knowledge. Reference the source document and page "
    "number when relevant. If the provided pages don't contain the answer, "
    "say so directly instead of guessing."
)

def build_multimodal_input(query_text: str, pages: list) -> list:
    content = [{"type": "text", "text": f"Question: {query_text}"}]
    for p in pages:
        with open(p["image_path"], "rb") as f:
            image_bytes = f.read()
        content.append({
            "type": "image",
            "data": base64.b64encode(image_bytes).decode("utf-8"),
            "mime_type": "image/png",
        })
    return content

import time
import random
import re as _re

# Gemini 3.6 Flash free tier (check aistudio.google.com/rate-limit for your
# project's actual numbers): RPM=5, RPD=20. Pacing below targets the RPM
# limit; RPD is a hard daily wall that no amount of waiting-and-retrying
# within a single run can get past -- see _daily_quota_exhausted below.
MIN_SECONDS_BETWEEN_CALLS = 13  # >60/5s, keeps comfortably under RPM=5
MAX_RETRIES = 3                 # per-minute throttles clear fast; a daily cap never will within a run

_last_call_time = [0.0]
_daily_quota_exhausted = [False]  # circuit breaker: set once, stops burning further attempts this run

def _extract_retry_after_seconds(error_text: str):
    """Gemini sometimes reports a suggested retry delay inside the 429 body,
    e.g. "retryDelay": "18s". Use it directly when present."""
    m = _re.search(r'"retryDelay"\s*:\s*"(\d+)s"', error_text)
    return int(m.group(1)) if m else None

def _looks_like_daily_quota(error_text: str) -> bool:
    """Heuristic: Gemini's 429 body names the exceeded quota metric --
    daily caps mention "PerDay", per-minute throttles mention "PerMinute".
    If neither phrase is present, treat it as transient (retry) rather
    than assuming the worst case."""
    lowered = error_text.lower()
    return "perday" in lowered or "per_day" in lowered

def generate_answer(query_text: str, pages: list) -> str:
    if _daily_quota_exhausted[0]:
        return "[DAILY_QUOTA_EXHAUSTED] skipped -- already hit the daily cap earlier this run"

    elapsed = time.time() - _last_call_time[0]
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)

    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            interaction = gemini_client.interactions.create(
                model=GEMINI_MODEL,
                input=build_multimodal_input(query_text, pages),
                system_instruction=SYSTEM_INSTRUCTION,
            )
            _last_call_time[0] = time.time()
            return interaction.output_text
        except Exception as e:
            last_error = e
            error_text = str(e)
            _last_call_time[0] = time.time()

            if "429" not in error_text:
                break  # not a rate-limit error -- retrying identically will not help

            if _looks_like_daily_quota(error_text):
                _daily_quota_exhausted[0] = True
                return f"[DAILY_QUOTA_EXHAUSTED] {error_text}"

            suggested = _extract_retry_after_seconds(error_text)
            wait = suggested if suggested is not None else min(30, (2 ** attempt) * 5)
            wait += random.uniform(0, 2)  # jitter, avoids retry pile-ups
            print(f"    [rate limited -- waiting {wait:.0f}s before retry {attempt + 1}/{MAX_RETRIES}]")
            time.sleep(wait)

    return f"[Gemini request failed after {MAX_RETRIES} attempts: {last_error}]"


Enter your Gemini API key (from https://aistudio.google.com/apikey): ··········


In [18]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9.% ]", " ", text)
    text = re.sub(r" +", " ", text)
    return text.strip()

def extract_keyphrases(gold_answer):
    """Pull out the load-bearing tokens from a gold answer: numbers/percentages
    and capitalized proper nouns. Good enough for this factual QA set --
    each gold_answer was written to have exactly 1-2 unambiguous key facts."""
    numbers = re.findall(r"\d+(?:\.\d+)?%?", gold_answer)
    proper_nouns = re.findall(r"\b[A-Z][a-zA-Z-]+\b", gold_answer)
    seen = []
    for kp in numbers + proper_nouns:
        if kp not in seen:
            seen.append(kp)
    return seen if seen else [gold_answer]

def keyword_grounded_score(model_answer, gold_answer):
    norm_answer = normalize(model_answer)
    keyphrases = extract_keyphrases(gold_answer)
    hits = 0
    for kp in keyphrases:
        if normalize(kp) in norm_answer:
            hits += 1
    return hits >= max(1, len(keyphrases) // 2)  # majority of key facts present


# ---- Optional: LLM-judge stub for later, more nuanced scoring ----
def llm_judge_score(question, gold_answer, model_answer):
    """Not called by default -- wire this in if keyword matching feels too
    blunt. Asks Gemini itself to grade correctness and faithfulness. Costs
    one extra API call per question."""
    prompt = (
        "Question: " + question + "\n"
        "Reference answer: " + gold_answer + "\n"
        "Model answer: " + model_answer + "\n\n"
        "Rate the model answer on two axes, each 0 or 1:\n"
        "correctness: does it match the reference answer's key facts?\n"
        "faithfulness: is it grounded in specific claims rather than vague/generic?\n"
        "Respond as JSON with keys correctness and faithfulness, each 0 or 1."
    )
    interaction = gemini_client.interactions.create(model=GEMINI_MODEL, input=prompt)
    return interaction.output_text  # parse as JSON in caller


### Run generation on top-1 retrieved page

Uses the actual top-1 page your retrieval step returned above -- not the gold page -- so this measures the *end-to-end* system (a wrong retrieval will typically also produce a wrong answer, which is the honest, realistic number). Change `TOP_K_FOR_GENERATION` to feed Gemini more than one page if you want to see whether extra context recovers from near-miss retrievals.

In [19]:
import json as _json_ckpt

from getpass import getpass

def swap_key() -> bool:
    """Called when a quota error survives all retries. Paste a key from a
    DIFFERENT Google Cloud project to continue; leave blank to stop."""
    global gemini_client
    new_key = getpass("Quota error. Paste a key from a different project (blank = stop): ").strip()
    if not new_key:
        _daily_quota_exhausted[0] = True   # skip the remaining questions instead of retrying each one
        return False
    gemini_client = genai.Client(api_key=new_key)
    _daily_quota_exhausted[0] = False
    return True

# Resumable: since the free tier caps at ~20 requests/day, this run will very
# likely stop partway through. Progress is saved after every single question,
# so re-running this exact cell later (after the daily reset, or with billing
# enabled) picks up only the remaining questions instead of starting over.
CHECKPOINT_PATH = "/content/drive/MyDrive/Visual-Doc/eval_results/eval1_generation_checkpoint.json"
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def _load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            return _json_ckpt.load(f)
    return {}

def _save_checkpoint(results_by_idx):
    with open(CHECKPOINT_PATH, "w") as f:
        _json_ckpt.dump(results_by_idx, f, indent=2)

def _is_real_failure(model_answer: str) -> bool:
    """True if this result should be retried on a future run (rate-limited /
    quota-exhausted), False if it's a completed answer worth keeping."""
    return model_answer.startswith("[Gemini request failed") or model_answer.startswith("[DAILY_QUOTA_EXHAUSTED]")

TOP_K_FOR_GENERATION = 1  # pages handed to Gemini per question

results_by_idx = _load_checkpoint()
already_done = sum(1 for v in results_by_idx.values() if not _is_real_failure(v["model_answer"]))
print(f"Resuming from checkpoint: {already_done}/{len(QA_SET)} questions already answered.")
print()

for i, qa in enumerate(QA_SET):
    key = str(i)
    doc_name = qa["doc_name"]
    page = qa["page"]

    if key in results_by_idx and not _is_real_failure(results_by_idx[key]["model_answer"]):
        print(f"[{i+1:>2}/{len(QA_SET)}] {doc_name:<24} p.{page:<4} -> (already done, skipping)")
        continue

    if _daily_quota_exhausted[0]:
        print(f"[{i+1:>2}/{len(QA_SET)}] {doc_name:<24} p.{page:<4} -> skipped (daily quota already hit this run)")
        continue

    ranked = exhaustive_maxsim_search(qa["question"], top_k=TOP_K_FOR_GENERATION)
    pages_for_gemini = [{"image_path": r["image_path"]} for r in ranked]

    model_answer = generate_answer(qa["question"], pages_for_gemini)
    while (model_answer.startswith("[DAILY_QUOTA_EXHAUSTED]")
          or (model_answer.startswith("[Gemini request failed") and "429" in model_answer)) and swap_key():
        model_answer = generate_answer(qa["question"], pages_for_gemini)
    correct = keyword_grounded_score(model_answer, qa["gold_answer"])

    if TOP_K_FOR_GENERATION == 1:
        retrieved_gold = retrieval_results[i]["hit_at_1"]
    else:
        gold_pair = (qa["doc_name"], qa["page"])
        retrieved_gold = any((r["doc_name"], r["page_number"]) == gold_pair for r in ranked)

    results_by_idx[key] = {
        "idx": i,
        "doc_name": doc_name,
        "gold_page": page,
        "modality": qa["modality"],
        "question": qa["question"],
        "gold_answer": qa["gold_answer"],
        "model_answer": model_answer,
        "retrieved_correct_page": retrieved_gold,
        "generation_correct": correct,
    }
    _save_checkpoint(results_by_idx)  # persist after every question -- safe to stop anytime

    if model_answer.startswith("[DAILY_QUOTA_EXHAUSTED]"):
        mark = "DAILY QUOTA HIT -- stopping early"
    elif _is_real_failure(model_answer):
        mark = "failed (will retry next run)"
    else:
        mark = "correct" if correct else "WRONG"
    print(f"[{i+1:>2}/{len(QA_SET)}] {doc_name:<24} p.{page:<4} -> {mark}")

completed = sum(1 for v in results_by_idx.values() if not _is_real_failure(v["model_answer"]))
print()
print(f"Progress: {completed}/{len(QA_SET)} questions answered so far.")
if completed < len(QA_SET):
    print("Some questions are still pending (rate-limited or quota-exhausted).")
    print("Just re-run this cell later (after the daily reset, or with billing")
    print("enabled) -- it will pick up only the remaining questions.")

# Only feed genuinely-answered questions into the results used below; pending
# ones are reported above, not silently counted as wrong.
generation_results = [
    results_by_idx[str(i)] for i in range(len(QA_SET))
    if str(i) in results_by_idx and not _is_real_failure(results_by_idx[str(i)]["model_answer"])
]


Resuming from checkpoint: 30/30 questions already answered.

[ 1/30] 2407.01449v6             p.1    -> (already done, skipping)
[ 2/30] 2407.01449v6             p.2    -> (already done, skipping)
[ 3/30] 2407.01449v6             p.4    -> (already done, skipping)
[ 4/30] 2407.01449v6             p.5    -> (already done, skipping)
[ 5/30] 2407.01449v6             p.7    -> (already done, skipping)
[ 6/30] 2407.01449v6             p.9    -> (already done, skipping)
[ 7/30] 2407.01449v6             p.18   -> (already done, skipping)
[ 8/30] 2407.01449v6             p.19   -> (already done, skipping)
[ 9/30] 2407.01449v6             p.21   -> (already done, skipping)
[10/30] 2407.01449v6             p.22   -> (already done, skipping)
[11/30] WorldDevelopmentReport   p.36   -> (already done, skipping)
[12/30] WorldDevelopmentReport   p.78   -> (already done, skipping)
[13/30] WorldDevelopmentReport   p.83   -> (already done, skipping)
[14/30] WorldDevelopmentReport   p.128  -> (already don

### Aggregate generation accuracy -- overall, per document, per modality, and split by whether retrieval actually found the gold page

In [20]:
gdf = pd.DataFrame(generation_results)

print("=== Overall generation accuracy ===")
acc = gdf["generation_correct"].mean()
n_correct = gdf["generation_correct"].sum()
n_total = len(gdf)
print(f"{acc:.1%}  ({n_correct}/{n_total})")

print()
print("=== By document ===")
display(gdf.groupby("doc_name")["generation_correct"].agg(["size", "mean"]))

print()
print("=== By modality ===")
display(gdf.groupby("modality")["generation_correct"].agg(["size", "mean"]))

print()
print("=== Generation accuracy conditioned on retrieval correctness ===")
print("(Low accuracy despite correct retrieval points at a generation/prompting")
print(" problem; low accuracy WITH wrong retrieval is a retrieval problem")
print(" masquerading as a generation problem.)")
display(gdf.groupby("retrieved_correct_page")["generation_correct"].agg(["size", "mean"]))

print()
print("=== Wrong answers (inspect these) ===")
wrong = gdf[~gdf["generation_correct"]]
if len(wrong):
    cols = ["doc_name", "gold_page", "modality", "question", "gold_answer", "model_answer", "retrieved_correct_page"]
    display(wrong[cols])
else:
    print("None -- every answer was scored correct by the keyword-grounded check.")


=== Overall generation accuracy ===
96.7%  (29/30)

=== By document ===


,size,mean
doc_name,,
2407.01449v6,10,0.9
AIFoundations,10,1.0
WorldDevelopmentReport,10,1.0



=== By modality ===


,size,mean
modality,,
chart,15,1.000000
diagram,2,1.000000
infographic,1,1.000000
map,2,1.000000
table,7,0.857143
text,3,1.000000



=== Generation accuracy conditioned on retrieval correctness ===
(Low accuracy despite correct retrieval points at a generation/prompting
 problem; low accuracy WITH wrong retrieval is a retrieval problem
 masquerading as a generation problem.)


,size,mean
retrieved_correct_page,,
False,4,0.75
True,26,1.00



=== Wrong answers (inspect these) ===


,doc_name,gold_page,modality,question,gold_answer,model_answer,retrieved_correct_page
4,2407.01449v6,7,table,"According to Table 2, what nDCG@5 score does C...",83.9,"Based on the provided document page, Table 2 i...",False


## 7b. Generation scoring audit, faithfulness and source-page accuracy

The Section 7 keyword scorer is lenient: it extracts capitalised words and numbers from the gold answer and passes an answer if **half** of them appear. An answer like *"Table 2 is not included on this page"* can pass on generic words such as "document" or "retrieval". Evaluation 2 will reuse this generation scoring for both arms, so it has to be trustworthy first. This section adds, without changing Section 7:

1. **Strict scorer** (`eval_metrics.strict_keyword_score`) -- fails any abstention ("not on this page"), requires *all* numbers in the gold answer, then applies the original keyword check.
2. **LLM judge** -- Gemini grades each answer against the gold answer **and the page image it was shown**, returning `correct`, `faithful` (every claim supported by that page; an honest abstention counts as faithful) and `abstained`. It costs 30 extra Gemini calls, is checkpointed and resumable, and reuses the same pacing and key-swap logic. It is the same model family as the generator, so treat it as a second opinion rather than ground truth.
3. **Manual overrides** -- the cell prints only the answers where graders disagree or the model abstained; record any verdict you want to override. This is the hand-grading the mid-sem slide refers to.
4. **Final verdict** = manual override > LLM judge > strict scorer, with the source of each verdict recorded.

**Source-page accuracy.** Gemini receives anonymous page images, so it cannot cite page numbers itself. The page the user sees comes from retrieval metadata, so source-page accuracy is whether the gold page is among the pages displayed (top-1 = what Gemini read; top-3 = the app default). "Correct answer **and** correct page" is the strictest end-to-end number.

In [21]:
import json as _json_j
import re, time, random, base64, io
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass

CALL_TIMEOUT = 120        # seconds to wait for one grading reply (overloaded servers stall rather than answer)
PREFLIGHT_TIMEOUT = 40   # seconds for the tiny key check (a one-word reply should take a few seconds)
JUDGE_MODEL = "gemini-3.5-flash-lite"   # different from the generator (gemini-3.6-flash); recorded per answer
                                        # fallback if its quota runs out: "gemini-3.1-flash-lite"
RUN_LLM_JUDGE = True      # 30 Gemini calls in total (26 may already be in the checkpoint). False = skip
PREFLIGHT = True          # one tiny text-only request first: does the ACTIVE key respond at all?

JUDGE_CHECKPOINT = "/content/drive/MyDrive/Visual-Doc/eval_results/eval1_judge_checkpoint.json"

JUDGE_SYSTEM = (
    "You are a strict grader for a document question-answering evaluation. You are given a question, "
    "a reference answer, a model answer, and the page image the model was shown. Grade the model answer.\n"
    "correct: 1 if the model answer states the reference answer's key fact(s) -- paraphrase, units and "
    "formatting differences are fine; 0 if it is wrong, incomplete on the key fact, or says the page "
    "does not contain the answer.\n"
    "faithful: 1 if every claim in the model answer is supported by the page image shown (an honest "
    "'the page does not contain the answer' counts as faithful); 0 if it asserts something the page "
    "does not support.\n"
    "abstained: 1 if the model answer says the provided page does not contain the answer, else 0.\n"
    "Respond with ONLY a JSON object: {\"correct\": 0|1, \"faithful\": 0|1, \"abstained\": 0|1, "
    "\"reason\": \"<one short sentence>\"}"
)

_call_pool = ThreadPoolExecutor(max_workers=8)   # a hung request occupies one worker; the cell never blocks
_user_gave_up = [False]                          # set when you leave a key prompt blank -> the run stops

# ------------------------------------------------------------------ API keys
_k = getpass("Paste ONE Gemini API key to start with (blank = keep the key already active): ").strip().strip("\"'")
if _k:
    gemini_client = genai.Client(api_key=_k)
    _daily_quota_exhausted[0] = False
    _last_call_time[0] = 0.0
    print("Now using the key you pasted. If it fails, you will be asked for the next one.")
else:
    print("Keeping the active key. If it fails, you will be asked for a new one.")
# (No key-format check: Google now issues keys starting 'AQ.' as well as the older 'AIza' ones.)

def _is_quota_error(text):
    t = text.lower()
    return any(m in t for m in ("429", "resource_exhausted", "quota", "rate limit", "rate_limit", "too many requests"))

def _is_billing_error(text):
    """403 'free tier is not available for this project': that key can never work without billing."""
    t = text.lower()
    return any(m in t for m in ("free tier is not available", "set up billing", "permissiondeniederror",
                                "permission_denied", "error code: 403"))

def _is_timeout_error(text):
    t = text.lower()
    return "timeouterror" in t or "timed out" in t

def _is_transient_error(text):
    t = text.lower()
    return any(m in t for m in ("503", "unavailable", "overloaded", "500", "internal", "deadline"))

def _is_bad_key_error(text):
    t = text.lower()
    return any(m in t for m in ("api_key_invalid", "api key not valid", "api key expired",
                                "api key was reported", "invalid api key", "api key is invalid"))

def _prompt_new_key(reason):
    """Ask for a replacement key. Blank input = stop trying."""
    global gemini_client
    k = getpass(f"{reason} Paste the next Gemini API key (blank = stop): ").strip().strip("\"'")
    if not k:
        _user_gave_up[0] = True
        return False
    gemini_client = genai.Client(api_key=k)
    _daily_quota_exhausted[0] = False
    _last_call_time[0] = 0.0
    return True

def _reason_for(raw):
    if _is_billing_error(raw):
        return "This key's project cannot use the free tier (billing required)."
    if _is_bad_key_error(raw):
        return "API key rejected as invalid."
    if _is_quota_error(raw):
        return "Quota exceeded."
    return "This key is not responding."

# ------------------------------------------------------------------ the Gemini call
def _create_with_timeout(parts, system_instruction, timeout=None):
    """Run the call in a worker thread; give up after CALL_TIMEOUT seconds instead of freezing the cell."""
    client = gemini_client                        # capture now so a later key switch can't affect this call
    kwargs = {"model": JUDGE_MODEL or GEMINI_MODEL, "input": parts}
    if system_instruction:
        kwargs["system_instruction"] = system_instruction
    fut = _call_pool.submit(lambda: client.interactions.create(**kwargs))
    return fut.result(timeout=timeout or CALL_TIMEOUT)

def _judge_gemini_call(parts, system_instruction=JUDGE_SYSTEM, timeout=None):
    """Same pacing logic as generate_answer. Quota -> short waits; temporary 5xx -> back-off retries;
    a timeout is returned at once (the caller decides what to do); anything else fails immediately."""
    if _daily_quota_exhausted[0]:
        return "[DAILY_QUOTA_EXHAUSTED] skipped -- already hit the daily cap earlier this run"
    elapsed = time.time() - _last_call_time[0]
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)
    last_error, n_tried = None, 0
    for attempt in range(MAX_RETRIES):
        n_tried = attempt + 1
        try:
            interaction = _create_with_timeout(parts, system_instruction, timeout)
            _last_call_time[0] = time.time()
            return interaction.output_text
        except Exception as e:
            last_error, error_text = e, f"{type(e).__name__}: {e}"
            _last_call_time[0] = time.time()
            if _is_timeout_error(error_text):
                return f"[Gemini request failed after {n_tried} attempt(s): TimeoutError: no reply within {timeout or CALL_TIMEOUT}s]"
            if _is_billing_error(error_text) or _is_bad_key_error(error_text):
                break                               # this key can't work: fail at once so the caller switches key
            if _is_quota_error(error_text):
                if _looks_like_daily_quota(error_text):
                    _daily_quota_exhausted[0] = True
                    return f"[DAILY_QUOTA_EXHAUSTED] {error_text}"
                suggested = _extract_retry_after_seconds(error_text)
                wait = (suggested if suggested is not None else min(30, (2 ** attempt) * 5)) + random.uniform(0, 2)
                print(f"    [rate limited -- waiting {wait:.0f}s before retry {attempt + 1}/{MAX_RETRIES}]")
                time.sleep(wait)
            elif _is_transient_error(error_text):
                wait = (15, 45, 90)[min(attempt, 2)] + random.uniform(0, 2)       # overload spikes need real waits
                print(f"    [server overloaded / temporary error -- waiting {wait:.0f}s before retry {attempt + 1}/{MAX_RETRIES}]")
                time.sleep(wait)
            else:
                break
    return f"[Gemini request failed after {n_tried} attempt(s): {type(last_error).__name__}: {last_error}]"

def _to01(v):
    if isinstance(v, str):
        return int(v.strip().lower() in ("1", "true", "yes"))
    return int(bool(v))

def _parse_judge(text):
    if not isinstance(text, str):
        return None
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", t, flags=re.S)
    if not m:
        return None
    try:
        d = _json_j.loads(m.group(0))
        return {"correct": _to01(d["correct"]), "faithful": _to01(d["faithful"]),
                "abstained": _to01(d.get("abstained", 0)), "reason": str(d.get("reason", ""))}
    except Exception:
        return None

_page_img = {(c["doc_name"], c["page_number"]): c["image_path"] for c in CORPUS}

def _make_parts(prompt, path, compact=False):
    """compact=True sends a smaller JPEG copy of the page (used as a fallback if the full PNG stalls)."""
    if compact:
        img = Image.open(path).convert("RGB")
        img.thumbnail((1600, 1600))
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=85)
        data, mime = base64.b64encode(buf.getvalue()).decode("utf-8"), "image/jpeg"
    else:
        with open(path, "rb") as f:
            data, mime = base64.b64encode(f.read()).decode("utf-8"), "image/png"
    return [{"type": "text", "text": prompt}, {"type": "image", "data": data, "mime_type": mime}]

def _load_judge_ckpt():
    if os.path.exists(JUDGE_CHECKPOINT):
        with open(JUDGE_CHECKPOINT) as f:
            return _json_j.load(f)
    return {}

judge_by_idx = _load_judge_ckpt()
_n_done = sum(1 for v in judge_by_idx.values() if v.get("correct") is not None)
_proceed = RUN_LLM_JUDGE and _n_done < len(generation_results)

# ------------------------------------------------------------------ preflight: does the active key respond?
if _proceed and PREFLIGHT:
    _user_gave_up[0] = False
    while True:
        t0 = time.time()
        raw = _judge_gemini_call([{"type": "text", "text": "Reply with the single word OK."}],
                                 system_instruction=None, timeout=PREFLIGHT_TIMEOUT)
        if not _is_real_failure(raw):
            print(f"Preflight: the active key responds ({time.time() - t0:.1f}s).\n")
            break
        print(f"Preflight FAILED: {raw[:250]}")
        if not _prompt_new_key(_reason_for(raw)):
            print("\nNo working key -- stopping. Re-run this cell and paste a working key; it resumes from the checkpoint.")
            _proceed = False
            break

# ------------------------------------------------------------------ grade the remaining answers
if RUN_LLM_JUDGE and _proceed:
    print(f"Judge checkpoint: {_n_done}/{len(generation_results)} already graded.\n")
    consecutive_errors = 0
    _user_gave_up[0] = False
    for g in generation_results:
        i, key = g["idx"], str(g["idx"])
        if key in judge_by_idx and judge_by_idx[key].get("correct") is not None:
            continue
        top1 = (retrieval_results[i]["top1_doc"], retrieval_results[i]["top1_page"])
        prompt = (f"Question: {g['question']}\nReference answer: {g['gold_answer']}\n"
                  f"Model answer: {g['model_answer']}\n\nGrade the model answer.")
        parts = _make_parts(prompt, _page_img[top1])
        raw = _judge_gemini_call(parts)
        if _is_real_failure(raw) and _is_timeout_error(raw):
            print(f"    [no reply within {CALL_TIMEOUT}s -> retrying once with a smaller JPEG copy of the page]")
            parts = _make_parts(prompt, _page_img[top1], compact=True)
            raw = _judge_gemini_call(parts)
        # quota problem, invalid key OR billing-required key -> switch to the next key (or ask for one) and retry this same question
        while _is_real_failure(raw) and (_is_billing_error(raw) or _is_bad_key_error(raw) or _is_quota_error(raw)) and \
                _prompt_new_key(_reason_for(raw)):
            raw = _judge_gemini_call(parts)
        api_failed = _is_real_failure(raw)
        parsed = None if api_failed else _parse_judge(raw)
        judge_by_idx[key] = {"idx": i, "judge_model": JUDGE_MODEL or GEMINI_MODEL, "raw": raw, **(parsed or {"correct": None, "faithful": None,
                                                                   "abstained": None, "reason": ""})}
        os.makedirs(os.path.dirname(JUDGE_CHECKPOINT), exist_ok=True)
        with open(JUDGE_CHECKPOINT, "w") as f:
            _json_j.dump(judge_by_idx, f, indent=2)
        if parsed is not None:
            consecutive_errors = 0
            print(f"[{i+1:>2}/{len(QA_SET)}] {g['doc_name']:<24} p.{g['gold_page']:<4} -> "
                  f"correct={parsed['correct']} faithful={parsed['faithful']}")
            continue
        consecutive_errors += 1
        why = "API error" if api_failed else "reply was not valid judge JSON"
        print(f"[{i+1:>2}/{len(QA_SET)}] {g['doc_name']:<24} p.{g['gold_page']:<4} -> FAILED ({why})")
        print(f"       {raw[:300]!r}")
        if _user_gave_up[0]:
            print("\nStopped: no key available. Re-run this cell with a working key; it resumes from the checkpoint.")
            break
        if consecutive_errors >= 2:             # two in a row: offer a different key instead of burning more time
            if not _prompt_new_key("Two requests in a row failed."):
                print("\nStopped. Re-run this cell with a working key; it resumes from the checkpoint.")
                break
            consecutive_errors = 0
elif not RUN_LLM_JUDGE:
    print("RUN_LLM_JUDGE = False -- skipping; faithfulness will be unavailable below.")

n_judged = sum(1 for v in judge_by_idx.values() if v.get("correct") is not None)
print(f"\nJudge progress: {n_judged}/{len(generation_results)} graded.")
if n_judged < len(generation_results) and RUN_LLM_JUDGE:
    print("Re-run this cell to continue -- it asks for a key each time and resumes from the checkpoint.")
    print("If the failures were 503 / 'high demand', the model is overloaded for every key: wait and re-run, set")
    print("JUDGE_MODEL to another model, or set RUN_LLM_JUDGE = False -- the remaining answers then fall back to")
    print("the strict scorer plus your manual review (faithfulness is reported over the answers that were judged).")

Paste ONE Gemini API key to start with (blank = keep the key already active): ··········
Now using the key you pasted. If it fails, you will be asked for the next one.

Judge progress: 30/30 graded.


In [22]:
# Per-answer verdicts from every grader, side by side.
audit = gdf.copy()
audit["keyword_orig"] = audit["generation_correct"]
audit["abstained"] = audit["model_answer"].map(is_abstention)
audit["strict"] = [strict_keyword_score(a, g) for a, g in zip(audit["model_answer"], audit["gold_answer"])]
audit["judge_correct"] = audit["idx"].map(lambda i: judge_by_idx.get(str(i), {}).get("correct"))
audit["judge_faithful"] = audit["idx"].map(lambda i: judge_by_idx.get(str(i), {}).get("faithful"))
audit["judge_reason"] = audit["idx"].map(lambda i: judge_by_idx.get(str(i), {}).get("reason", ""))

def _disagrees(r):
    votes = [bool(r["keyword_orig"]), bool(r["strict"])]
    if pd.notna(r["judge_correct"]):
        votes.append(bool(r["judge_correct"]))
    return len(set(votes)) > 1 or bool(r["abstained"])

audit["needs_review"] = audit.apply(_disagrees, axis=1)

print(f"Original keyword scorer : {int(audit['keyword_orig'].sum())}/{len(audit)} correct")
print(f"Strict scorer           : {int(audit['strict'].sum())}/{len(audit)} correct")
if audit["judge_correct"].notna().any():
    print(f"LLM judge               : {int(audit['judge_correct'].fillna(0).sum())}/{int(audit['judge_correct'].notna().sum())} correct")
print(f"Answers that abstain    : {int(audit['abstained'].sum())}")
print(f"Answers needing review  : {int(audit['needs_review'].sum())}  (graders disagree, or the model abstained)")
print(f"Not graded by the judge : {int(audit['judge_correct'].isna().sum())}  (shown below so you can check them yourself)\n")

REVIEW_ALL = False       # True = print all answers, not just the ones that need review
_show = audit if REVIEW_ALL else audit[audit["needs_review"] | audit["judge_correct"].isna()]   # also show answers the judge has not graded
for _, r in _show.iterrows():
    print("-" * 100)
    print(f"idx {r['idx']}  |  {r['doc_name']} p.{r['gold_page']}  |  {r['modality']}  |  "
          f"retrieved gold page: {r['retrieved_correct_page']}")
    print(f"Q   : {r['question']}")
    print(f"GOLD: {r['gold_answer']}")
    print(f"ANS : {r['model_answer'][:400]}")
    print(f"      keyword={r['keyword_orig']}  strict={r['strict']}  judge={r['judge_correct']}  "
          f"abstained={r['abstained']}  | judge reason: {r['judge_reason']}")

Original keyword scorer : 29/30 correct
Strict scorer           : 24/30 correct
LLM judge               : 28/30 correct
Answers that abstain    : 3
Answers needing review  : 6  (graders disagree, or the model abstained)
Not graded by the judge : 0  (shown below so you can check them yourself)

----------------------------------------------------------------------------------------------------
idx 3  |  2407.01449v6 p.5  |  chart  |  retrieved gold page: False
Q   : According to Figure 2, what is the total per-page offline indexing latency for SigLIP?
GOLD: 0.12 seconds
ANS : Based on the provided document page (page 19), **Figure 2 is not present**. 

*(Note: Table 5 on this page lists the total per-page indexing latency for SigLIP as **0.12 seconds**, but Figure 2 itself is located on a different page not included here.)*
      keyword=True  strict=False  judge=1  abstained=True  | judge reason: The model correctly provides the latency of 0.12 seconds from Table 5 while noting Figure 

In [23]:
import json

# Your hand-grading. After reading the answers printed above, record any verdict you want to
# OVERRIDE as {idx: True/False}. A manual verdict always beats the automatic ones; anything you
# leave out falls back to the LLM judge, then to the strict scorer. Saved to Drive, so it survives restarts.
MANUAL_OVERRIDES = {
    3: False,    # p5, Figure 2: said the figure is not on its page; the 0.12 s value came from Table 5 on a different page
    4: False,    # p7, Table 2: said the table is not on its page
    27: False,   # p108, Figure 5.8: listed several countries instead of naming one
}

MANUAL_PATH = "/content/drive/MyDrive/Visual-Doc/eval_results/eval1_manual_grades.json"
_saved = {}
if os.path.exists(MANUAL_PATH):
    with open(MANUAL_PATH) as f:
        _saved = {int(k): bool(v) for k, v in json.load(f).items()}
_saved.update({int(k): bool(v) for k, v in MANUAL_OVERRIDES.items()})
os.makedirs(os.path.dirname(MANUAL_PATH), exist_ok=True)
with open(MANUAL_PATH, "w") as f:
    json.dump({str(k): v for k, v in _saved.items()}, f, indent=2)
MANUAL_GRADES = _saved
print(f"{len(MANUAL_GRADES)} manual verdict(s) in effect: {MANUAL_GRADES}")

3 manual verdict(s) in effect: {3: False, 4: False, 27: False}


In [24]:
def _final(r):
    if r["idx"] in MANUAL_GRADES:
        return bool(MANUAL_GRADES[r["idx"]]), "manual"
    if pd.notna(r["judge_correct"]):
        return bool(r["judge_correct"]), "llm_judge"
    return bool(r["strict"]), "strict"

_fv = audit.apply(_final, axis=1)
audit["final_correct"] = [v[0] for v in _fv]
audit["verdict_source"] = [v[1] for v in _fv]
audit["hit_at_1"] = audit["idx"].map(lambda i: retrieval_results[i]["hit_at_1"]).astype(bool)
audit["hit_at_3"] = audit["idx"].map(lambda i: retrieval_results[i]["hit_at_3"]).astype(bool)
audit["rank_of_gold"] = audit["idx"].map(lambda i: retrieval_results[i]["rank_of_gold"])
final_df = audit

n = len(final_df)
print(f"Verdict sources: {final_df['verdict_source'].value_counts().to_dict()}\n")
print("=== Generation accuracy (final verdicts) ===")
print(f"Overall                         : {final_df['final_correct'].mean():.1%}  ({int(final_df['final_correct'].sum())}/{n})")
print(f"  when gold page was top-1      : {final_df.loc[final_df.hit_at_1, 'final_correct'].mean():.1%}  "
      f"({int(final_df.loc[final_df.hit_at_1, 'final_correct'].sum())}/{int(final_df.hit_at_1.sum())})")
if (~final_df.hit_at_1).any():
    print(f"  when retrieval missed rank 1  : {final_df.loc[~final_df.hit_at_1, 'final_correct'].mean():.1%}  "
          f"({int(final_df.loc[~final_df.hit_at_1, 'final_correct'].sum())}/{int((~final_df.hit_at_1).sum())})")

print("\n=== Source-page accuracy ===")
print("Gemini sees anonymous page images, so it cannot cite page numbers itself; the page shown to the")
print("user comes from retrieval metadata. Source-page accuracy therefore = is the gold page among the")
print("pages displayed?")
print(f"  top-1 page is the gold page   : {final_df['hit_at_1'].mean():.1%}  (the page Gemini actually read)")
print(f"  gold page in top-3 shown      : {final_df['hit_at_3'].mean():.1%}  (app default k=3)")
grounded = final_df["final_correct"] & final_df["hit_at_1"]
print(f"  correct answer AND correct page: {grounded.mean():.1%}  ({int(grounded.sum())}/{n})  <- end-to-end, evidence-backed")

if final_df["judge_faithful"].notna().any():
    jf = final_df["judge_faithful"].dropna()
    print("\n=== Faithfulness (LLM judge, answer vs. the page Gemini was shown) ===")
    print(f"Faithful answers                : {jf.mean():.1%}  ({int(jf.sum())}/{len(jf)})  -- judged {len(jf)} of {n} answers")
    unfaithful = final_df[final_df["judge_faithful"] == 0]
    if len(unfaithful):
        print("Unfaithful answers (inspect):", unfaithful["idx"].tolist())

miss = final_df[~final_df["hit_at_1"]]
if len(miss):
    print("\n=== Behaviour when retrieval missed rank 1 ===")
    print(f"{len(miss)} questions: {int(miss['abstained'].sum())} abstained ('not on this page'), "
          f"{int((~miss['abstained']).sum())} answered anyway")
    print(f"  of those answered anyway, correct: {int(miss.loc[~miss['abstained'], 'final_correct'].sum())}"
          f"/{int((~miss['abstained']).sum())}")

print("\n=== By document ===")
display(final_df.groupby("doc_name")["final_correct"].agg(["size", "mean"]))
print("=== By modality ===")
display(final_df.groupby("modality")["final_correct"].agg(["size", "mean"]))

Verdict sources: {'llm_judge': 27, 'manual': 3}

=== Generation accuracy (final verdicts) ===
Overall                         : 90.0%  (27/30)
  when gold page was top-1      : 100.0%  (26/26)
  when retrieval missed rank 1  : 25.0%  (1/4)

=== Source-page accuracy ===
Gemini sees anonymous page images, so it cannot cite page numbers itself; the page shown to the
user comes from retrieval metadata. Source-page accuracy therefore = is the gold page among the
pages displayed?
  top-1 page is the gold page   : 86.7%  (the page Gemini actually read)
  gold page in top-3 shown      : 96.7%  (app default k=3)
  correct answer AND correct page: 86.7%  (26/30)  <- end-to-end, evidence-backed

=== Faithfulness (LLM judge, answer vs. the page Gemini was shown) ===
Faithful answers                : 100.0%  (30/30)  -- judged 30 of 30 answers

=== Behaviour when retrieval missed rank 1 ===
4 questions: 3 abstained ('not on this page'), 1 answered anyway
  of those answered anyway, correct: 1/1

==

,size,mean
doc_name,,
2407.01449v6,10,0.8
AIFoundations,10,0.9
WorldDevelopmentReport,10,1.0


=== By modality ===


,size,mean
modality,,
chart,15,0.866667
diagram,2,1.000000
infographic,1,1.000000
map,2,1.000000
table,7,0.857143
text,3,1.000000


## 8. Save results

In [25]:
OUTPUT_DIR = "/content/drive/MyDrive/Visual-Doc/eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.to_csv(os.path.join(OUTPUT_DIR, "eval1_retrieval_results.csv"), index=False)
gdf.to_csv(os.path.join(OUTPUT_DIR, "eval1_generation_results.csv"), index=False)

summary = {
    "retrieval": {
        "overall": recall_table(df).to_dict("records")[0],
        "by_document": recall_table(df, "doc_name").to_dict("index"),
        "by_modality": recall_table(df, "modality").to_dict("index"),
    },
    "generation": {
        "overall_accuracy": float(gdf["generation_correct"].mean()),
        "by_document": gdf.groupby("doc_name")["generation_correct"].mean().to_dict(),
        "by_modality": gdf.groupby("modality")["generation_correct"].mean().to_dict(),
    },
}
import json as json_module
with open(os.path.join(OUTPUT_DIR, "eval1_summary.json"), "w") as f:
    json_module.dump(summary, f, indent=2)

print(f"Saved to {OUTPUT_DIR}/")
print("  - eval1_retrieval_results.csv   (per-question retrieval detail)")
print("  - eval1_generation_results.csv  (per-question generation detail)")
print("  - eval1_summary.json            (aggregated headline numbers)")


Saved to /content/drive/MyDrive/Visual-Doc/eval_results/
  - eval1_retrieval_results.csv   (per-question retrieval detail)
  - eval1_generation_results.csv  (per-question generation detail)
  - eval1_summary.json            (aggregated headline numbers)


## 8b. Final results and freeze for Evaluation 2

Writes the final per-question table and summary, and **freezes** everything Evaluation 2 must reuse so the only variable there is the retrieval method:

- `benchmark_v1/qa_set.json` -- the exact questions, with a hash
- `benchmark_v1/eval_config.json` -- generator model, system prompt, pages per question, retrieval and metric settings, environment
- `benchmark_v1/visual_arm_ranks.json` -- ColPali + MaxSim ranks per question, so Evaluation 2 can compare against them without re-running the model

The original Section 8 cell is left as it was; it records the keyword-scorer numbers. The `*_final*` files here use the audited verdicts.

Also records library versions, the judge model(s) actually used, the manual overrides, and 95% Wilson intervals for Recall@1 and generation accuracy. If 6c/6d were not run, it prints a warning and writes `null` for those parts instead of skipping silently.

In [26]:
import json, hashlib, platform, datetime, math
import importlib.metadata as _im

def _jsonable(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (set, tuple)): return list(o)
    if isinstance(o, float) and np.isnan(o): return None
    return str(o)

def _sanitize(o):
    if isinstance(o, dict): return {str(k): _sanitize(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [_sanitize(v) for v in o]
    o = _jsonable(o) if isinstance(o, (np.generic, np.ndarray)) else o
    if isinstance(o, float) and np.isnan(o): return None
    return o

def wilson(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n."""
    if n == 0: return (None, None)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def _ver(pkg):
    try: return _im.version(pkg)
    except Exception: return None

VERSIONS = {p: _ver(p) for p in ("colpali-engine", "transformers", "peft", "torch", "chromadb",
                                 "google-genai", "numpy", "pandas", "pillow")}

FROZEN_DIR = os.path.join(OUTPUT_DIR, "benchmark_v1")
os.makedirs(FROZEN_DIR, exist_ok=True)

# warn loudly instead of silently writing nulls
_missing = [n for n in ("EFFICIENCY", "ABLATION") if n not in globals()]
if _missing:
    print(f"NOTE: {_missing} not found -- run Sections 6c/6d first. They will be written as null.")

# 1) the benchmark itself + a hash
qa_canonical = json.dumps(QA_SET, sort_keys=True, ensure_ascii=False)
qa_hash = hashlib.sha256(qa_canonical.encode("utf-8")).hexdigest()[:16]
with open(os.path.join(FROZEN_DIR, "qa_set.json"), "w") as f:
    json.dump({"qa_hash": qa_hash, "n_questions": len(QA_SET), "questions": QA_SET}, f, indent=2, ensure_ascii=False)

# judge models actually used (the checkpoint may mix models)
judge_models = {}
for v in judge_by_idx.values():
    judge_models[v.get("judge_model")] = judge_models.get(v.get("judge_model"), 0) + 1

# 2) frozen generation / retrieval settings
eval_config = {
    "qa_hash": qa_hash,
    "generation": {"model": GEMINI_MODEL, "system_instruction": SYSTEM_INSTRUCTION,
                   "pages_per_question": TOP_K_FOR_GENERATION, "input": "page image(s) as PNG, question as text"},
    "judge": {"models_used": judge_models, "note": "same family as the generator; second opinion only"},
    "visual_arm_retrieval": {"model": MODEL_NAME, "mode": "exhaustive MaxSim over all pages, no doc filter",
                             "pages_indexed": len(CORPUS), "docs": DOC_NAME_COUNTS,
                             "retrieval_device": "cpu (Section 6 ran on CPU; bf16 numerics can differ on GPU)"},
    "metrics": {"recall_ks": [1, 3, 5, 10], "mrr": "MRR@10", "ndcg": "nDCG@5 (single relevant page)",
                "module": "eval_metrics.py"},
    "answer_verdict_rule": "manual override > LLM judge > strict keyword scorer",
    "manual_overrides": {str(k): v for k, v in MANUAL_GRADES.items()},
    "environment": {"device": DEVICE,
                    "gpu": EFFICIENCY["gpu"] if "EFFICIENCY" in globals() else None,
                    "python": platform.python_version(), "versions": VERSIONS,
                    "date": datetime.date.today().isoformat()},
}
with open(os.path.join(FROZEN_DIR, "eval_config.json"), "w") as f:
    json.dump(_sanitize(eval_config), f, indent=2, default=_jsonable)

# 3) the visual arm's ranks
visual_arm = [{"idx": r["idx"], "doc_name": r["doc_name"], "gold_page": r["gold_page"],
               "rank_of_gold": r["rank_of_gold"], "top10": r["top10"]} for r in retrieval_results]
with open(os.path.join(FROZEN_DIR, "visual_arm_ranks.json"), "w") as f:
    json.dump(_sanitize({"qa_hash": qa_hash, "arm": "ColPali + exhaustive MaxSim", "results": visual_arm}),
              f, indent=2, default=_jsonable)

# 4) final per-question table + final summary
final_cols = ["idx", "doc_name", "gold_page", "modality", "question", "gold_answer", "rank_of_gold",
              "hit_at_1", "hit_at_3", "model_answer", "keyword_orig", "strict", "abstained",
              "judge_correct", "judge_faithful", "judge_reason", "final_correct", "verdict_source"]
final_df[final_cols].to_csv(os.path.join(OUTPUT_DIR, "eval1_final_per_question.csv"), index=False)

_ok = final_df["final_correct"]
n = len(final_df)
ci = lambda k: dict(zip(("lo", "hi"), wilson(int(k), n)))
final_summary = {
    "qa_hash": qa_hash,
    "retrieval": {"overall": metrics_table(df).to_dict("records")[0],
                  "by_document": metrics_table(df, "doc_name").to_dict("index"),
                  "by_modality": metrics_table(df, "modality").to_dict("index"),
                  "wilson95": {"Recall@1": ci(df["hit_at_1"].sum()), "Recall@3": ci(df["hit_at_3"].sum()),
                               "Recall@10": ci(df["hit_at_10"].sum())}},
    "retrieval_mode_ablation": ABLATION.to_dict("index") if "ABLATION" in globals() else None,
    "efficiency": EFFICIENCY if "EFFICIENCY" in globals() else None,
    "generation": {
        "n_answered": int(n),
        "accuracy_final": float(_ok.mean()),
        "accuracy_final_wilson95": ci(_ok.sum()),
        "accuracy_original_keyword_scorer": float(final_df["keyword_orig"].mean()),
        "accuracy_strict_scorer": float(final_df["strict"].mean()),
        "accuracy_llm_judge": float(final_df["judge_correct"].dropna().mean()) if final_df["judge_correct"].notna().any() else None,
        "verdict_sources": final_df["verdict_source"].value_counts().to_dict(),
        "judge_coverage": {"judged": int(final_df["judge_correct"].notna().sum()), "of": int(n)},
        "accuracy_when_gold_top1": float(_ok[final_df.hit_at_1].mean()),
        "faithfulness_llm_judge": float(final_df["judge_faithful"].dropna().mean()) if final_df["judge_faithful"].notna().any() else None,
        "by_document": final_df.groupby("doc_name")["final_correct"].mean().to_dict(),
        "by_modality": final_df.groupby("modality")["final_correct"].mean().to_dict(),
    },
    "source_page_accuracy": {"top1": float(final_df["hit_at_1"].mean()), "top3": float(final_df["hit_at_3"].mean()),
                             "correct_answer_and_correct_page": float((_ok & final_df.hit_at_1).mean())},
}
with open(os.path.join(OUTPUT_DIR, "eval1_summary_final.json"), "w") as f:
    json.dump(_sanitize(final_summary), f, indent=2, default=_jsonable)

print(f"Benchmark hash: {qa_hash}  ({len(QA_SET)} questions)")
print(f"Judge models used: {judge_models}")
print(f"Verdict sources  : {final_df['verdict_source'].value_counts().to_dict()}")
print(f"Final accuracy   : {int(_ok.sum())}/{n} = {_ok.mean():.1%}  (Wilson 95%: {wilson(int(_ok.sum()), n)[0]:.1%}-{wilson(int(_ok.sum()), n)[1]:.1%})")
lo, hi = wilson(int(df['hit_at_1'].sum()), len(df))
print(f"Recall@1         : {int(df['hit_at_1'].sum())}/{len(df)}  (Wilson 95%: {lo:.1%}-{hi:.1%})")
print(f"\nSaved to {OUTPUT_DIR}/ and frozen in {FROZEN_DIR}/")
print("  - eval1_final_per_question.csv, eval1_summary_final.json")
print("  - benchmark_v1/qa_set.json, eval_config.json (with library versions), visual_arm_ranks.json")
print("  - ../eval_metrics.py is the shared scoring code both arms must import")
if _missing:
    print(f"\nREMINDER: {_missing} were not available, so those parts of the summary are null.")


Benchmark hash: 6057af19ccde7e1c  (30 questions)
Judge models used: {None: 26, 'gemini-3.5-flash-lite': 4}
Verdict sources  : {'llm_judge': 27, 'manual': 3}
Final accuracy   : 27/30 = 90.0%  (Wilson 95%: 74.4%-96.5%)
Recall@1         : 26/30  (Wilson 95%: 70.3%-94.7%)

Saved to /content/drive/MyDrive/Visual-Doc/eval_results/ and frozen in /content/drive/MyDrive/Visual-Doc/eval_results/benchmark_v1/
  - eval1_final_per_question.csv, eval1_summary_final.json
  - benchmark_v1/qa_set.json, eval_config.json (with library versions), visual_arm_ranks.json
  - ../eval_metrics.py is the shared scoring code both arms must import


## Next: Evaluations 2 and 3

This notebook only covers **Evaluation 1** (system evaluation, full late interaction only). It deliberately does not touch:

- **Evaluation 2 (baseline comparison)** -- would add an OCR/text-RAG arm (e.g., BM25 or BGE-M3 over text extracted from these same PDFs) run against the *same 30 questions*, so retrieval method is the only variable. `df` and `QA_SET` above are already in the right shape to extend with a second retrieval column once that arm exists.
- **Evaluation 3 (real-world comparison)** -- would run the same 30 questions against a general-purpose assistant (e.g., ChatGPT with file upload) for a qualitative comparison point.

Both are separate follow-on notebooks, not additions to this one.

**Hand-off to Evaluation 2.** Section 8b freezes the questions, generator settings and the ColPali arm's ranks in `eval_results/benchmark_v1/`, and `eval_metrics.py` holds the shared metric and scoring code. Evaluation 2 should load those files, build the OCR/text arm over the same page PNGs (mapping every text chunk back to its `(doc_name, page)`), and score it with `eval_metrics` so retrieval method is the only variable.